# TEMA 2 IA 


In [ ]:
import os
import sys
import json
import time
import math
import random
import shutil
import zipfile
import warnings
import subprocess
import importlib.util
import copy
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

if importlib.util.find_spec("albumentations") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

import torchvision
from torchvision import transforms, models
from torchvision.utils import make_grid

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_AVAILABLE = True
except Exception:
    TENSORBOARD_AVAILABLE = False
    print("TensorBoard nu este disponibil. Continuăm fără TensorBoard.")

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_AMP = DEVICE.type == "cuda"
try:
    from torch.amp import autocast, GradScaler
    AMP_SCALER = GradScaler("cuda", enabled=USE_AMP)
    def amp_autocast():
        return autocast(device_type="cuda", enabled=USE_AMP)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    AMP_SCALER = GradScaler(enabled=USE_AMP)
    def amp_autocast():
        return autocast(enabled=USE_AMP)

IS_KAGGLE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
PROJECT_ROOT = WORK_DIR
DATA_DIR = WORK_DIR / "data"
ARTIFACTS_DIR = WORK_DIR / "artifacts"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"

for d in [DATA_DIR, ARTIFACTS_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_WORKERS = min(2, os.cpu_count() or 2) if IS_KAGGLE else 0
PIN_MEMORY = DEVICE.type == "cuda"

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Albumentations version:", A.__version__)
print("Device:", DEVICE)
print("Kaggle:", IS_KAGGLE)
print("WORK_DIR:", WORK_DIR)
print("NUM_WORKERS:", NUM_WORKERS)
print("Mixed precision:", USE_AMP)
print("TensorBoard available:", TENSORBOARD_AVAILABLE)

In [ ]:
MOLE_ZIP = Path.cwd() / "vai-de-pielea-mea.zip"
FACE_ZIP = Path.cwd() / "you-re-on-candid-camera.zip"


def extract_zip_if_needed(zip_path: Path, target_dir: Path):
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"Folder existent: {target_dir}")
        return
    if not zip_path.exists():
        raise FileNotFoundError(
            f"Nu găsesc arhiva {zip_path.name}. Local pune fișierul lângă notebook; "
            f"pe Kaggle adaugă dataset-ul în /kaggle/input."
        )
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"Dezarhivez {zip_path.name} în {target_dir}...")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(target_dir)
    print("Dezarhivare terminată.")


def find_first_file(root: Path, filename: str) -> Path:
    found = sorted(root.rglob(filename))
    if not found:
        raise FileNotFoundError(f"Nu am găsit {filename} în {root}")
    return found[0]


def find_first_folder(root: Path, folder_name: str) -> Path:
    found = sorted([p for p in root.rglob(folder_name) if p.is_dir()])
    if not found:
        raise FileNotFoundError(f"Nu am găsit folderul {folder_name} în {root}")
    return found[0]


def maybe_file(*candidates: Path) -> Optional[Path]:
    for p in candidates:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def maybe_folder(*candidates: Path) -> Optional[Path]:
    for p in candidates:
        if p is not None and Path(p).exists() and Path(p).is_dir():
            return Path(p)
    return None


def locate_dataset_root(folder_name: str, local_target: Path, zip_path: Path) -> Path:
    """Găsește folderul datasetului pe Kaggle sau îl dezarhivează local."""
    if IS_KAGGLE:
        # Cazuri uzuale: /kaggle/input/<dataset>/dataset-tema2/<folder>
        priority = [
            Path("/kaggle/input/datasets/sebitopa/dataset-tema2") / folder_name,
            Path("/kaggle/input/dataset-tema2") / folder_name,
            Path("/kaggle/input/tema2-ia/dataset-tema2") / folder_name,
        ]
        for p in priority:
            if p.exists():
                return p
        matches = sorted([p for p in Path("/kaggle/input").rglob(folder_name) if p.is_dir()])
        if matches:
            return matches[0]
        raise FileNotFoundError(
            f"Nu găsesc folderul {folder_name} în /kaggle/input. "
            f"Adaugă datasetul Tema 2 la notebook-ul Kaggle."
        )
    else:
        extract_zip_if_needed(zip_path, local_target)
        
        nested = maybe_folder(local_target / folder_name)
        return nested if nested is not None else local_target


MOLE_ROOT = locate_dataset_root(
    "vai-de-pielea-mea",
    DATA_DIR / "vai-de-pielea-mea",
    MOLE_ZIP
)
FACE_ROOT = locate_dataset_root(
    "you-re-on-candid-camera",
    DATA_DIR / "you-re-on-candid-camera",
    FACE_ZIP
)


MOLE_TRAIN_CSV = maybe_file(MOLE_ROOT / "train.csv") or find_first_file(MOLE_ROOT, "train.csv")
MOLE_TEST_CSV = maybe_file(MOLE_ROOT / "test.csv") or find_first_file(MOLE_ROOT, "test.csv")
MOLE_KAGGLE_TEST_CSV = maybe_file(MOLE_ROOT / "kaggle_test.csv")

MOLE_TRAIN_DIR = maybe_folder(MOLE_ROOT / "train") or find_first_folder(MOLE_ROOT, "train")
MOLE_TEST_DIR = maybe_folder(MOLE_ROOT / "test") or find_first_folder(MOLE_ROOT, "test")
MOLE_KAGGLE_TEST_DIR = maybe_folder(MOLE_ROOT / "kaggle_test")


FACE_SPLITS_DIR = maybe_folder(FACE_ROOT / "splits") or FACE_ROOT
FACE_DATASET_DIR = maybe_folder(FACE_ROOT / "DATASET") or FACE_ROOT

FACE_TRAIN_CSV = maybe_file(FACE_SPLITS_DIR / "train.csv", FACE_ROOT / "train.csv") or find_first_file(FACE_ROOT, "train.csv")
FACE_LOCAL_TEST_CSV = maybe_file(FACE_SPLITS_DIR / "local_test.csv", FACE_ROOT / "local_test.csv") or find_first_file(FACE_ROOT, "local_test.csv")
FACE_REMOTE_TEST_CSV = maybe_file(FACE_SPLITS_DIR / "remote_test.csv", FACE_ROOT / "remote_test.csv")
FACE_NORM_STATS = maybe_file(FACE_SPLITS_DIR / "norm_stats.json", FACE_ROOT / "norm_stats.json") or find_first_file(FACE_ROOT, "norm_stats.json")

df_mole_train = pd.read_csv(MOLE_TRAIN_CSV)
df_mole_test = pd.read_csv(MOLE_TEST_CSV)
df_mole_kaggle = pd.read_csv(MOLE_KAGGLE_TEST_CSV) if MOLE_KAGGLE_TEST_CSV is not None else None

df_face_train = pd.read_csv(FACE_TRAIN_CSV)
df_face_test = pd.read_csv(FACE_LOCAL_TEST_CSV)
df_face_remote = pd.read_csv(FACE_REMOTE_TEST_CSV) if FACE_REMOTE_TEST_CSV is not None else None

print("=== PATHS DETECTATE ===")
print("MOLE_ROOT:", MOLE_ROOT)
print("FACE_ROOT:", FACE_ROOT)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print()

print("=== VAI DE PIELEA MEA ===")
print("Train:", df_mole_train.shape, MOLE_TRAIN_CSV)
print("Test:", df_mole_test.shape, MOLE_TEST_CSV)
print("Kaggle test:", None if df_mole_kaggle is None else df_mole_kaggle.shape, MOLE_KAGGLE_TEST_CSV)
display(df_mole_train.head())

print("\n=== YOU'RE ON CANDID CAMERA ===")
print("Train:", df_face_train.shape, FACE_TRAIN_CSV)
print("Local test:", df_face_test.shape, FACE_LOCAL_TEST_CSV)
print("Remote test:", None if df_face_remote is None else df_face_remote.shape, FACE_REMOTE_TEST_CSV)
display(df_face_train.head())

In [ ]:
def detect_label_column(df: pd.DataFrame, possible_names):
    # 1. Încercăm întâi nume clasice de coloană
    for col in df.columns:
        if col.lower() in [name.lower() for name in possible_names]:
            return col

    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        if 2 <= n_unique <= 20:
            return col

    raise ValueError(f"Nu pot detecta coloana de label. Coloane disponibile: {list(df.columns)}")


def detect_path_column(df: pd.DataFrame, label_col: str):
    possible_names = [
        "path", "filepath", "file_path", "image_path",
        "img_path", "filename", "file_name", "image",
        "image_id", "id", "img"
    ]

    for col in df.columns:
        if col != label_col and col.lower() in possible_names:
            return col

    
    for col in df.columns:
        if col != label_col:
            return col

    raise ValueError("Nu pot detecta coloana cu imaginea.")


MOLE_LABEL_COL = detect_label_column(
    df_mole_train,
    possible_names=["diagnostic", "dx", "label", "class", "target"]
)

MOLE_PATH_COL = detect_path_column(df_mole_train, MOLE_LABEL_COL)

FACE_LABEL_COL = detect_label_column(
    df_face_train,
    possible_names=["label", "emotion", "class", "target"]
)

FACE_PATH_COL = detect_path_column(df_face_train, FACE_LABEL_COL)

print("Coloane detectate:")
print("MOLE_LABEL_COL:", MOLE_LABEL_COL)
print("MOLE_PATH_COL:", MOLE_PATH_COL)
print("FACE_LABEL_COL:", FACE_LABEL_COL)
print("FACE_PATH_COL:", FACE_PATH_COL)

In [ ]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

MOLE_CLASS_NAMES = {
    "akiec": "Bowen disease",
    "bcc": "Carcinom bazocelular",
    "bkl": "Keratoză benignă",
    "df": "Dermatofibrom",
    "mel": "Melanom",
    "nv": "Nevi melanocitari",
    "vasc": "Leziuni vasculare",
}

FACE_CLASS_NAMES = {
    1: "Surprise",
    2: "Fear",
    3: "Disgust",
    4: "Happiness",
    5: "Sadness",
    6: "Anger",
    7: "Neutral",
}

MOLE_LABEL_COL = df_mole_train.columns[1]
FACE_LABEL_COL = df_face_train.columns[1]

print("Coloană label mole:", MOLE_LABEL_COL)
print("Coloană label face:", FACE_LABEL_COL)

mole_counts = df_mole_train[MOLE_LABEL_COL].value_counts()
face_counts = df_face_train[FACE_LABEL_COL].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    mole_counts.index.astype(str),
    mole_counts.values,
    edgecolor="white"
)

axes[0].set_title("Vai de pielea mea — distribuție clase train", fontsize=13)
axes[0].set_xlabel("Clasă")
axes[0].set_ylabel("Număr imagini")

for i, value in enumerate(mole_counts.values):
    axes[0].text(i, value + 20, str(value), ha="center", fontsize=9)

face_labels = []

for cls in face_counts.index:
    cls_int = int(cls)
    face_labels.append(FACE_CLASS_NAMES.get(cls_int, str(cls_int)))

axes[1].bar(
    face_labels,
    face_counts.values,
    edgecolor="white"
)

axes[1].set_title("You're on candid camera — distribuție clase train", fontsize=13)
axes[1].set_xlabel("Emoție")
axes[1].set_ylabel("Număr imagini")
axes[1].tick_params(axis="x", rotation=30)

for i, value in enumerate(face_counts.values):
    axes[1].text(i, value + 30, str(value), ha="center", fontsize=9)

plt.tight_layout()

output_path = ARTIFACTS_DIR / "distributie_clase_train.png"
plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Grafic salvat în:", output_path)

In [ ]:
def collect_image_sizes(root_dir: Path, max_images: int = 100) -> pd.DataFrame:
    """
    Citește dimensiunile pentru cel mult max_images imagini dintr-un folder.
    Returnează un DataFrame cu lățime, înălțime și cale.
    """
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
    image_paths = []

    for pattern in extensions:
        image_paths.extend(root_dir.rglob(pattern))

    image_paths = image_paths[:max_images]

    records = []

    for image_path in image_paths:
        try:
            with Image.open(image_path) as img:
                width, height = img.size

            records.append({
                "path": str(image_path),
                "width": width,
                "height": height,
            })

        except Exception as err:
            print(f"Nu pot citi imaginea {image_path}: {err}")

    return pd.DataFrame(records)


mole_size_df = collect_image_sizes(MOLE_TRAIN_DIR, max_images=100)

face_train_image_dir = FACE_DATASET_DIR / "train"

if not face_train_image_dir.exists():
    face_train_image_dir = FACE_DATASET_DIR

face_size_df = collect_image_sizes(face_train_image_dir, max_images=100)


print("=== VAI DE PIELEA MEA — dimensiuni imagini train ===")
display(mole_size_df[["width", "height"]].describe().round(2))

print("\n=== YOU'RE ON CANDID CAMERA — dimensiuni imagini train ===")
display(face_size_df[["width", "height"]].describe().round(2))


size_summary = {
    "mole": mole_size_df[["width", "height"]].describe().round(2).to_dict(),
    "face": face_size_df[["width", "height"]].describe().round(2).to_dict(),
}

with open(ARTIFACTS_DIR / "image_size_summary.json", "w", encoding="utf-8") as f:
    json.dump(size_summary, f, indent=2, ensure_ascii=False)

print("\nSumarul dimensiunilor a fost salvat în:", ARTIFACTS_DIR / "image_size_summary.json")

In [ ]:
def load_normalization_stats(json_path: Path) -> Tuple[List[float], List[float]]:
    """
    Încarcă mean/std din fișierul norm_stats.json.
    Acceptă mai multe variante posibile de chei: mean/means și std/stds.
    """
    with open(json_path, "r", encoding="utf-8") as file:
        stats = json.load(file)

    mean = stats["mean"] if "mean" in stats else stats["means"]
    std = stats["std"] if "std" in stats else stats["stds"]

    return mean, std


SKIN_MEAN = [0.7646, 0.5464, 0.5712]
SKIN_STD = [0.1410, 0.1532, 0.1706]

CAMERA_MEAN, CAMERA_STD = load_normalization_stats(FACE_NORM_STATS)


normalization_table = pd.DataFrame({
    "dataset": ["Vai de pielea mea", "You're on candid camera"],
    "mean_R": [SKIN_MEAN[0], CAMERA_MEAN[0]],
    "mean_G": [SKIN_MEAN[1], CAMERA_MEAN[1]],
    "mean_B": [SKIN_MEAN[2], CAMERA_MEAN[2]],
    "std_R": [SKIN_STD[0], CAMERA_STD[0]],
    "std_G": [SKIN_STD[1], CAMERA_STD[1]],
    "std_B": [SKIN_STD[2], CAMERA_STD[2]],
})

display(normalization_table)

normalization_table.to_csv(
    ARTIFACTS_DIR / "normalization_stats.csv",
    index=False
)

print("Statisticile de normalizare au fost salvate în:")
print(ARTIFACTS_DIR / "normalization_stats.csv")

In [ ]:
def get_existing_image_path(root_dir: Path, value) -> Path:
    """
    Primește valoarea din CSV și încearcă să găsească imaginea reală.
    Merge pentru căi relative, nume de fișier sau id-uri simple.
    """
    value = str(value)
    candidate = root_dir / value

    if candidate.exists():
        return candidate

    candidate = root_dir / Path(value).name

    if candidate.exists():
        return candidate

    stem = Path(value).stem

    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        matches = list(root_dir.rglob(stem + ext))
        if matches:
            return matches[0]

    matches = list(root_dir.rglob(Path(value).name))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Nu am găsit imaginea: {value}")


def class_name_for_plot(class_id, names_dict: dict) -> str:
    """
    Returnează numele clasei pentru titlul imaginii.
    """
    if class_id in names_dict:
        return names_dict[class_id]

    try:
        class_id_int = int(class_id)
        return names_dict.get(class_id_int, str(class_id))
    except Exception:
        return str(class_id)


def plot_original_images_grid(
    df: pd.DataFrame,
    image_root: Path,
    label_column: str,
    image_column: str,
    class_names: dict,
    title: str,
    output_filename: str,
    samples_per_class: int = 4
):
    """
    Afișează o grilă cu imagini originale:
    fiecare rând = o clasă,
    fiecare coloană = un exemplu din clasa respectivă.
    """
    classes = sorted(df[label_column].dropna().unique())

    fig, axes = plt.subplots(
        nrows=len(classes),
        ncols=samples_per_class,
        figsize=(samples_per_class * 2.6, len(classes) * 2.4)
    )

    if len(classes) == 1:
        axes = np.expand_dims(axes, axis=0)

    fig.suptitle(title, fontsize=15, y=1.01)

    for row_idx, class_id in enumerate(classes):
        class_df = df[df[label_column] == class_id]

        chosen_samples = class_df.sample(
            n=min(samples_per_class, len(class_df)),
            random_state=SEED
        )

        for col_idx in range(samples_per_class):
            ax = axes[row_idx, col_idx]
            ax.axis("off")

            if col_idx >= len(chosen_samples):
                continue

            row = chosen_samples.iloc[col_idx]
            img_path = get_existing_image_path(image_root, row[image_column])

            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)

            readable_name = class_name_for_plot(class_id, class_names)
            ax.set_title(f"{class_id} — {readable_name}", fontsize=8)

    plt.tight_layout()

    save_path = ARTIFACTS_DIR / output_filename
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Figura a fost salvată în: {save_path}")

In [ ]:
plot_original_images_grid(
    df=df_mole_train,
    image_root=MOLE_TRAIN_DIR,
    label_column=MOLE_LABEL_COL,
    image_column=MOLE_PATH_COL,
    class_names=MOLE_CLASS_NAMES,
    title="Exemple originale pe clase — Vai de pielea mea",
    output_filename="originale_mole.png",
    samples_per_class=4
)

plot_original_images_grid(
    df=df_face_train,
    image_root=FACE_DATASET_DIR,
    label_column=FACE_LABEL_COL,
    image_column=FACE_PATH_COL,
    class_names=FACE_CLASS_NAMES,
    title="Exemple originale pe clase — You're on candid camera",
    output_filename="originale_faces.png",
    samples_per_class=4
)

In [ ]:
import cv2

mole_aug = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.6),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=8, val_shift_limit=8, p=0.3),
    A.GaussianBlur(blur_limit=(3, 3), p=0.15),
])


face_aug = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.Affine(
        scale=(0.90, 1.10),
        translate_percent=(-0.05, 0.05),
        rotate=(-12, 12),
        shear=(-4, 4),
        p=0.8
    ),
    A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.6),
    A.GaussianBlur(blur_limit=(3, 3), p=0.15),
])

In [ ]:
def plot_augmented_examples_by_class(
    df: pd.DataFrame,
    image_root: Path,
    label_column: str,
    image_column: str,
    class_names: dict,
    augmentation,
    title: str,
    output_filename: str,
    n_versions: int = 3
):
    classes = sorted(df[label_column].dropna().unique())

    fig, axes = plt.subplots(
        nrows=len(classes),
        ncols=n_versions + 1,
        figsize=((n_versions + 1) * 3.0, len(classes) * 2.7)
    )

    if len(classes) == 1:
        axes = np.expand_dims(axes, axis=0)

    fig.suptitle(title, fontsize=15, y=1.01)

    for row_idx, class_id in enumerate(classes):
        class_df = df[df[label_column] == class_id]
        selected_row = class_df.sample(n=1, random_state=SEED).iloc[0]

        img_path = get_existing_image_path(image_root, selected_row[image_column])
        original = np.array(Image.open(img_path).convert("RGB"))

        images = [original]

        for _ in range(n_versions):
            aug_img = augmentation(image=original)["image"]
            images.append(aug_img)

        for col_idx, img in enumerate(images):
            ax = axes[row_idx, col_idx]
            ax.imshow(img)
            ax.axis("off")

            if row_idx == 0:
                if col_idx == 0:
                    ax.set_title("Original", fontsize=10)
                else:
                    ax.set_title(f"Augmentare {col_idx}", fontsize=10)

            if col_idx == 0:
                label_text = class_name_for_plot(class_id, class_names)
                ax.set_ylabel(
                    f"{class_id}\n{label_text}",
                    rotation=0,
                    labelpad=55,
                    va="center",
                    fontsize=9
                )

    plt.tight_layout()

    save_path = ARTIFACTS_DIR / output_filename
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Figura a fost salvată în: {save_path}")

In [ ]:
plot_augmented_examples_by_class(
    df=df_mole_train,
    image_root=MOLE_TRAIN_DIR,
    label_column=MOLE_LABEL_COL,
    image_column=MOLE_PATH_COL,
    class_names=MOLE_CLASS_NAMES,
    augmentation=mole_aug,
    title="3.1 Augmentări — Vai de pielea mea",
    output_filename="augmentari_mole.png",
    n_versions=3
)

plot_augmented_examples_by_class(
    df=df_face_train,
    image_root=FACE_DATASET_DIR,
    label_column=FACE_LABEL_COL,
    image_column=FACE_PATH_COL,
    class_names=FACE_CLASS_NAMES,
    augmentation=face_aug,
    title="3.1 Augmentări — You're on candid camera",
    output_filename="augmentari_faces.png",
    n_versions=3
)

In [ ]:
class ImageCSVDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        image_column: str,
        label_column: str,
        class_to_idx: dict,
        transform=None
    ):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.image_column = image_column
        self.label_column = label_column
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = get_existing_image_path(
            self.image_root,
            row[self.image_column]
        )

        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        label_raw = str(row[self.label_column])
        label = self.class_to_idx[label_raw]

        return image, label

In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def build_image_index_fast(root_dirs):
    index = {}

    for root in root_dirs:
        root = Path(root)

        if not root.exists():
            continue

        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMG_EXTENSIONS:
                index[path.name] = path
                index[path.stem] = path

                try:
                    index[str(path.relative_to(root))] = path
                except Exception:
                    pass

    return index


def resolve_image_path_fast(value, root_dirs, image_index):
    value = str(value)
    value_path = Path(value)

    if value_path.is_absolute() and value_path.exists():
        return value_path

    for root in root_dirs:
        root = Path(root)

        candidate = root / value
        if candidate.exists():
            return candidate

        candidate = root / value_path.name
        if candidate.exists():
            return candidate

    for key in [value, value_path.name, value_path.stem]:
        if key in image_index:
            return image_index[key]

    raise FileNotFoundError(f"Nu am găsit imaginea: {value}")


class FastImageCSVDataset(Dataset):
    def __init__(
        self,
        df,
        root_dirs,
        image_column,
        label_column,
        class_to_idx,
        transform=None
    ):
        self.df = df.reset_index(drop=True)
        self.root_dirs = [Path(r) for r in root_dirs]
        self.image_column = image_column
        self.label_column = label_column
        self.class_to_idx = class_to_idx
        self.transform = transform

        self.image_index = build_image_index_fast(self.root_dirs)
        self.samples = []

        for _, row in self.df.iterrows():
            img_path = resolve_image_path_fast(
                row[self.image_column],
                self.root_dirs,
                self.image_index
            )

            label = self.class_to_idx[str(row[self.label_column])]
            self.samples.append((img_path, label))

        print(f"Dataset creat: {len(self.samples)} imagini")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        return image, label

In [ ]:
MLP_IMAGE_SIZE = 48
BATCH_SIZE = 128
MLP_EPOCHS = 5

def make_basic_transform(mean, std, image_size=48):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


skin_mlp_transform = make_basic_transform(SKIN_MEAN, SKIN_STD, MLP_IMAGE_SIZE)
camera_mlp_transform = make_basic_transform(CAMERA_MEAN, CAMERA_STD, MLP_IMAGE_SIZE)


skin_class_to_idx = {
    cls: idx
    for idx, cls in enumerate(sorted(df_mole_train[MOLE_LABEL_COL].astype(str).unique()))
}

camera_class_to_idx = {
    cls: idx
    for idx, cls in enumerate(sorted(df_face_train[FACE_LABEL_COL].astype(str).unique()))
}

skin_idx_to_class = {v: k for k, v in skin_class_to_idx.items()}
camera_idx_to_class = {v: k for k, v in camera_class_to_idx.items()}


skin_train_mlp_ds = FastImageCSVDataset(
    df=df_mole_train,
    root_dirs=[MOLE_TRAIN_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_mlp_transform
)

skin_test_mlp_ds = FastImageCSVDataset(
    df=df_mole_test,
    root_dirs=[MOLE_TEST_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_mlp_transform
)

camera_train_mlp_ds = FastImageCSVDataset(
    df=df_face_train,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=camera_mlp_transform
)

camera_test_mlp_ds = FastImageCSVDataset(
    df=df_face_test,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=camera_mlp_transform
)


skin_train_mlp_loader = DataLoader(
    skin_train_mlp_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

skin_test_mlp_loader = DataLoader(
    skin_test_mlp_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

camera_train_mlp_loader = DataLoader(
    camera_train_mlp_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

camera_test_mlp_loader = DataLoader(
    camera_test_mlp_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


print("Clase skin:", skin_class_to_idx)
print("Clase camera:", camera_class_to_idx)

xb, yb = next(iter(skin_train_mlp_loader))
print("Batch skin:", xb.shape, yb.shape)

xb, yb = next(iter(camera_train_mlp_loader))
print("Batch camera:", xb.shape, yb.shape)

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        image_size: int = 48,
        use_dropout: bool = True,
        dropout_p: float = 0.30,
        use_batchnorm: bool = True
    ):
        super().__init__()

        input_dim = 3 * image_size * image_size

        layers = [
            nn.Flatten(),

            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512) if use_batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p) if use_dropout else nn.Identity(),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256) if use_batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p) if use_dropout else nn.Identity(),

            nn.Linear(256, num_classes)
        ]

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        if USE_AMP:
            AMP_SCALER.scale(loss).backward()
            AMP_SCALER.step(optimizer)
            AMP_SCALER.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
    }


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp_autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
    }

In [ ]:
def plot_training_curves(history: pd.DataFrame, title: str, output_name: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["val_loss"], label="test")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epocă")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(history["epoch"], history["train_f1_macro"], label="train")
    axes[1].plot(history["epoch"], history["val_f1_macro"], label="test")
    axes[1].set_title("F1-macro")
    axes[1].set_xlabel("Epocă")
    axes[1].set_ylabel("F1-macro")
    axes[1].legend()

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()

    save_path = ARTIFACTS_DIR / output_name
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Curbe salvate în:", save_path)


def plot_confusion_matrix_simple(y_true, y_pred, class_labels, title, output_name):
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicție")
    ax.set_ylabel("Etichetă reală")

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))

    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticklabels(class_labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.colorbar(im, ax=ax)
    plt.tight_layout()

    save_path = ARTIFACTS_DIR / output_name
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Matrice salvată în:", save_path)


def run_mlp_experiment(
    experiment_name: str,
    train_loader,
    test_loader,
    num_classes: int,
    class_labels: List[str],
    use_dropout: bool,
    epochs: int = 10,
    lr: float = 1e-3,
    weight_decay: float = 1e-4
):
    print(f"\n=== Experiment: {experiment_name} ===")

    model = MLPClassifier(
        num_classes=num_classes,
        image_size=MLP_IMAGE_SIZE,
        use_dropout=use_dropout,
        dropout_p=0.35,
        use_batchnorm=True
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=max(epochs // 2, 1),
        gamma=0.5
    )

    history_rows = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE
        )

        val_metrics = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=DEVICE
        )

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_f1_micro": train_metrics["f1_micro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        history_rows.append(row)

        print(
            f"Epoca {epoch:02d}/{epochs} | "
            f"train loss={row['train_loss']:.4f}, train F1={row['train_f1_macro']:.4f} | "
            f"test loss={row['val_loss']:.4f}, test F1={row['val_f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE
    )

    history = pd.DataFrame(history_rows)

    safe_name = experiment_name.replace(" ", "_").replace("/", "_").lower()

    history.to_csv(
        ARTIFACTS_DIR / f"{safe_name}_history.csv",
        index=False
    )

    plot_training_curves(
        history=history,
        title=f"Curbe train/test — {experiment_name}",
        output_name=f"{safe_name}_curves.png"
    )

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=class_labels,
        title=f"Matrice de confuzie — {experiment_name}",
        output_name=f"{safe_name}_confusion_matrix.png"
    )

    torch.save(
        model.state_dict(),
        ARTIFACTS_DIR / f"{safe_name}_best_model.pt"
    )

    result = {
        "experiment": experiment_name,
        "use_dropout": use_dropout,
        "best_val_f1_macro": best_f1,
        "final_accuracy": final_metrics["accuracy"],
        "final_f1_macro": final_metrics["f1_macro"],
        "final_f1_micro": final_metrics["f1_micro"],
        "final_loss": final_metrics["loss"],
    }

    print("\nRezultat final:")
    print(result)

    with open(ARTIFACTS_DIR / f"{safe_name}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return model, history, result

In [ ]:
MLP_EPOCHS = 5

skin_class_labels = [
    MOLE_CLASS_NAMES.get(skin_idx_to_class[i], skin_idx_to_class[i])
    for i in range(len(skin_idx_to_class))
]

mlp_skin_dropout_model, mlp_skin_dropout_history, mlp_skin_dropout_result = run_mlp_experiment(
    experiment_name="MLP skin cu dropout",
    train_loader=skin_train_mlp_loader,
    test_loader=skin_test_mlp_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    use_dropout=True,
    epochs=MLP_EPOCHS
)

mlp_skin_dropout_result

In [ ]:
CAMERA_MLP_SUBSET_SIZE = 3500   # poți crește la 5000 dacă merge repede
BATCH_SIZE_FAST = 256

camera_subset_indices = (
    df_face_train
    .groupby(FACE_LABEL_COL, group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), max(1, int(CAMERA_MLP_SUBSET_SIZE * len(x) / len(df_face_train)))),
        random_state=SEED
    ))
    .index
    .tolist()
)

camera_train_mlp_subset = Subset(camera_train_mlp_ds, camera_subset_indices)

camera_train_mlp_loader_fast = DataLoader(
    camera_train_mlp_subset,
    batch_size=BATCH_SIZE_FAST,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

camera_test_mlp_loader_fast = DataLoader(
    camera_test_mlp_ds,
    batch_size=BATCH_SIZE_FAST,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Camera train original:", len(camera_train_mlp_ds))
print("Camera train subset:", len(camera_train_mlp_subset))

In [ ]:
camera_class_labels = [
    FACE_CLASS_NAMES.get(int(camera_idx_to_class[i]), str(camera_idx_to_class[i]))
    for i in range(len(camera_idx_to_class))
]

print(camera_class_labels)

In [ ]:

MLP_EPOCHS = 4

mlp_camera_dropout_model, mlp_camera_dropout_history, mlp_camera_dropout_result = run_mlp_experiment(
    experiment_name="MLP camera cu dropout",
    train_loader=camera_train_mlp_loader_fast,
    test_loader=camera_test_mlp_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    use_dropout=True,
    epochs=MLP_EPOCHS
)

mlp_camera_dropout_result

In [ ]:

MLP_EPOCHS = 5

mlp_camera_no_dropout_model, mlp_camera_no_dropout_history, mlp_camera_no_dropout_result = run_mlp_experiment(
    experiment_name="MLP camera fara dropout",
    train_loader=camera_train_mlp_loader_fast,
    test_loader=camera_test_mlp_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    use_dropout=False,
    epochs=MLP_EPOCHS
)

mlp_camera_no_dropout_result

In [ ]:
MLP_EPOCHS = 5

mlp_skin_no_dropout_model, mlp_skin_no_dropout_history, mlp_skin_no_dropout_result = run_mlp_experiment(
    experiment_name="MLP skin fara dropout",
    train_loader=skin_train_mlp_loader,
    test_loader=skin_test_mlp_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    use_dropout=False,
    epochs=MLP_EPOCHS
)

mlp_skin_no_dropout_result

In [ ]:
CNN_IMAGE_SIZE = 64
CNN_BATCH_SIZE = 256
CNN_NUM_WORKERS = NUM_WORKERS

def make_cnn_transform(mean, std, image_size=64):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


skin_cnn_transform = make_cnn_transform(SKIN_MEAN, SKIN_STD, CNN_IMAGE_SIZE)
camera_cnn_transform = make_cnn_transform(CAMERA_MEAN, CAMERA_STD, CNN_IMAGE_SIZE)


skin_train_cnn_ds = FastImageCSVDataset(
    df=df_mole_train,
    root_dirs=[MOLE_TRAIN_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_cnn_transform
)

skin_test_cnn_ds = FastImageCSVDataset(
    df=df_mole_test,
    root_dirs=[MOLE_TEST_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_cnn_transform
)

camera_train_cnn_ds = FastImageCSVDataset(
    df=df_face_train,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=camera_cnn_transform
)

camera_test_cnn_ds = FastImageCSVDataset(
    df=df_face_test,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=camera_cnn_transform
)


skin_train_cnn_loader = DataLoader(
    skin_train_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    shuffle=True,
    num_workers=CNN_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

skin_test_cnn_loader = DataLoader(
    skin_test_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    shuffle=False,
    num_workers=CNN_NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

camera_train_cnn_loader = DataLoader(
    camera_train_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    shuffle=True,
    num_workers=CNN_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

camera_test_cnn_loader = DataLoader(
    camera_test_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    shuffle=False,
    num_workers=CNN_NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Device:", DEVICE)

xb, yb = next(iter(camera_train_cnn_loader))
print("Camera CNN batch:", xb.shape, yb.shape)

xb, yb = next(iter(skin_train_cnn_loader))
print("Skin CNN batch:", xb.shape, yb.shape)

In [ ]:
class SimpleCNNClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        first_kernel_size: int = 3,
        use_batchnorm: bool = True,
        dropout_p: float = 0.25
    ):
        super().__init__()

        padding_first = first_kernel_size // 2

        def conv_block(in_channels, out_channels, kernel_size=3, padding=1):
            layers = [
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    bias=not use_batchnorm
                )
            ]

            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))

            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(kernel_size=2))

            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(3, 16, kernel_size=first_kernel_size, padding=padding_first),
            conv_block(16, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )

    
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

      
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x



test_cnn = SimpleCNNClassifier(num_classes=7, first_kernel_size=3).to(DEVICE)
dummy = torch.randn(4, 3, CNN_IMAGE_SIZE, CNN_IMAGE_SIZE).to(DEVICE)

with torch.no_grad():
    out = test_cnn(dummy)

print("Output shape:", out.shape)

In [ ]:
def run_cnn_experiment(
    experiment_name: str,
    train_loader,
    test_loader,
    num_classes: int,
    class_labels: List[str],
    first_kernel_size: int = 3,
    use_batchnorm: bool = True,
    epochs: int = 5,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    class_weights=None,
):
    """Antrenează, evaluează și salvează un CNN pentru un experiment.

    Funcția este folosită de toate celulele CNN: baseline, ablație BatchNorm,
    ablație augmentări și WeightedRandomSampler.
    """
    print(f"\n=== Experiment CNN: {experiment_name} ===")

    model = SimpleCNNClassifier(
        num_classes=num_classes,
        first_kernel_size=first_kernel_size,
        use_batchnorm=use_batchnorm,
        dropout_p=0.25,
    ).to(DEVICE)

    if class_weights is not None:
        if not torch.is_tensor(class_weights):
            class_weights = torch.tensor(class_weights, dtype=torch.float32)
        class_weights = class_weights.to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
    )

    history_rows = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
        )

        val_metrics = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=DEVICE,
        )

        scheduler.step(val_metrics["f1_macro"])

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_f1_micro": train_metrics["f1_micro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history_rows.append(row)

        print(
            f"Epoca {epoch:02d}/{epochs} | "
            f"train loss={row['train_loss']:.4f}, train F1={row['train_f1_macro']:.4f} | "
            f"test loss={row['val_loss']:.4f}, test F1={row['val_f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE,
    )

    history = pd.DataFrame(history_rows)
    safe_name = experiment_name.replace(" ", "_").replace("/", "_").lower()

    history.to_csv(
        ARTIFACTS_DIR / f"{safe_name}_history.csv",
        index=False,
    )

    plot_training_curves(
        history=history,
        title=f"Curbe train/test — {experiment_name}",
        output_name=f"{safe_name}_curves.png",
    )

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=class_labels,
        title=f"Matrice de confuzie — {experiment_name}",
        output_name=f"{safe_name}_confusion_matrix.png",
    )

    torch.save(
        model.state_dict(),
        ARTIFACTS_DIR / f"{safe_name}_best_model.pt",
    )

    result = {
        "experiment": experiment_name,
        "model": "CNN",
        "first_kernel_size": first_kernel_size,
        "use_batchnorm": use_batchnorm,
        "best_val_f1_macro": best_f1,
        "final_accuracy": final_metrics["accuracy"],
        "final_f1_macro": final_metrics["f1_macro"],
        "final_f1_micro": final_metrics["f1_micro"],
        "final_loss": final_metrics["loss"],
    }

    print("\nRezultat final:")
    print(result)

    with open(ARTIFACTS_DIR / f"{safe_name}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return model, history, result


### 3.3 — CNN pe Vai de pielea mea

Rularea lungă duplicată a fost eliminată pentru Kaggle. Experimentul CNN skin cu BatchNorm este rulat mai jos o singură dată, cu `CNN_EPOCHS = 3`, ca să nu consume timp inutil.

In [ ]:
CNN_EPOCHS = 3

cnn_camera_model, cnn_camera_history, cnn_camera_result = run_cnn_experiment(
    experiment_name="CNN camera cu BatchNorm",
    train_loader=camera_train_cnn_loader,
    test_loader=camera_test_cnn_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_result

In [ ]:
CNN_EPOCHS = 3

cnn_skin_model, cnn_skin_history, cnn_skin_result = run_cnn_experiment(
    experiment_name="CNN skin cu BatchNorm",
    train_loader=skin_train_cnn_loader,
    test_loader=skin_test_cnn_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_skin_result

In [ ]:

FINETUNE_IMAGE_SIZE = 160   
FINETUNE_BATCH_SIZE = 64
FINETUNE_NUM_WORKERS = NUM_WORKERS

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def make_finetune_transform(image_size=160):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


finetune_transform = make_finetune_transform(FINETUNE_IMAGE_SIZE)


skin_train_ft_ds = FastImageCSVDataset(
    df=df_mole_train,
    root_dirs=[MOLE_TRAIN_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=finetune_transform
)

skin_test_ft_ds = FastImageCSVDataset(
    df=df_mole_test,
    root_dirs=[MOLE_TEST_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=finetune_transform
)

camera_train_ft_ds = FastImageCSVDataset(
    df=df_face_train,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=finetune_transform
)

camera_test_ft_ds = FastImageCSVDataset(
    df=df_face_test,
    root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
    image_column=FACE_PATH_COL,
    label_column=FACE_LABEL_COL,
    class_to_idx=camera_class_to_idx,
    transform=finetune_transform
)


skin_train_ft_loader = DataLoader(
    skin_train_ft_ds,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=True,
    num_workers=FINETUNE_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

skin_test_ft_loader = DataLoader(
    skin_test_ft_ds,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=FINETUNE_NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

camera_train_ft_loader = DataLoader(
    camera_train_ft_ds,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=True,
    num_workers=FINETUNE_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

camera_test_ft_loader = DataLoader(
    camera_test_ft_ds,
    batch_size=FINETUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=FINETUNE_NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

xb, yb = next(iter(skin_train_ft_loader))
print("Finetune skin batch:", xb.shape, yb.shape)

xb, yb = next(iter(camera_train_ft_loader))
print("Finetune camera batch:", xb.shape, yb.shape)

In [ ]:
def build_mobilenet_v2(num_classes: int, freeze_backbone: bool = True):
    """
    Încarcă MobileNetV2 preantrenat pe ImageNet și înlocuiește capul de clasificare.
    Dacă Kaggle nu are internet/cache pentru weights, notebook-ul continuă cu weights=None,
    dar pentru scor mai bun este recomandat să activezi Internet sau să atașezi weights/checkpoints.
    """
    try:
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
        print("MobileNetV2: weights ImageNet încărcate.")
    except Exception as exc:
        print("Atenție: nu am putut încărca weights ImageNet pentru MobileNetV2.")
        print("Motiv:", repr(exc))
        print("Continui cu weights=None ca notebook-ul să poată rula pe Kaggle offline.")
        model = models.mobilenet_v2(weights=None)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.30),
        nn.Linear(in_features, num_classes)
    )

    return model


def freeze_batchnorm_layers(model):
    """
    Pentru backbone înghețat, păstrăm BatchNorm în eval ca să nu modifice statisticile ImageNet.
    """
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            module.eval()

In [ ]:
def train_one_epoch_finetune(model, loader, criterion, optimizer, device, freeze_bn=True):
    model.train()

    if freeze_bn:
        freeze_batchnorm_layers(model)

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        if USE_AMP:
            AMP_SCALER.scale(loss).backward()
            AMP_SCALER.step(optimizer)
            AMP_SCALER.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
    }


@torch.no_grad()
def evaluate_finetune(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp_autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
    }

In [ ]:
def run_finetune_experiment(
    experiment_name: str,
    train_loader,
    test_loader,
    num_classes: int,
    class_labels: List[str],
    epochs: int = 4,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    freeze_backbone: bool = True
):
    print(f"\n=== Experiment fine-tuning: {experiment_name} ===")

    model = build_mobilenet_v2(
        num_classes=num_classes,
        freeze_backbone=freeze_backbone
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )

    history_rows = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch_finetune(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            freeze_bn=freeze_backbone
        )

        val_metrics = evaluate_finetune(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=DEVICE
        )

        scheduler.step(val_metrics["f1_macro"])

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_f1_micro": train_metrics["f1_micro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        history_rows.append(row)

        print(
            f"Epoca {epoch:02d}/{epochs} | "
            f"train loss={row['train_loss']:.4f}, train F1={row['train_f1_macro']:.4f} | "
            f"test loss={row['val_loss']:.4f}, test F1={row['val_f1_macro']:.4f} | "
            f"lr={row['lr']:.6f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_finetune(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE
    )

    history = pd.DataFrame(history_rows)

    safe_name = experiment_name.replace(" ", "_").replace("/", "_").lower()

    history.to_csv(
        ARTIFACTS_DIR / f"{safe_name}_history.csv",
        index=False
    )

    plot_training_curves(
        history=history,
        title=f"Curbe train/test — {experiment_name}",
        output_name=f"{safe_name}_curves.png"
    )

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=class_labels,
        title=f"Matrice de confuzie — {experiment_name}",
        output_name=f"{safe_name}_confusion_matrix.png"
    )

    torch.save(
        model.state_dict(),
        ARTIFACTS_DIR / f"{safe_name}_best_model.pt"
    )

    result = {
        "experiment": experiment_name,
        "model": "MobileNetV2 pretrained ImageNet",
        "freeze_backbone": freeze_backbone,
        "best_val_f1_macro": best_f1,
        "final_accuracy": final_metrics["accuracy"],
        "final_f1_macro": final_metrics["f1_macro"],
        "final_f1_micro": final_metrics["f1_micro"],
        "final_loss": final_metrics["loss"],
    }

    with open(ARTIFACTS_DIR / f"{safe_name}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nRezultat final:")
    print(result)

    return model, history, result

In [ ]:
FINETUNE_EPOCHS = 4

finetune_skin_model, finetune_skin_history, finetune_skin_result = run_finetune_experiment(
    experiment_name="MobileNetV2 skin freeze backbone",
    train_loader=skin_train_ft_loader,
    test_loader=skin_test_ft_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    epochs=FINETUNE_EPOCHS,
    lr=1e-3,
    freeze_backbone=True
)

finetune_skin_result

In [ ]:
FINETUNE_EPOCHS = 4

finetune_camera_model, finetune_camera_history, finetune_camera_result = run_finetune_experiment(
    experiment_name="MobileNetV2 camera freeze backbone",
    train_loader=camera_train_ft_loader,
    test_loader=camera_test_ft_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    epochs=FINETUNE_EPOCHS,
    lr=1e-3,
    freeze_backbone=True
)

finetune_camera_result

In [ ]:
CNN_EPOCHS = 3

cnn_skin_no_bn_model, cnn_skin_no_bn_history, cnn_skin_no_bn_result = run_cnn_experiment(
    experiment_name="CNN skin fara BatchNorm",
    train_loader=skin_train_cnn_loader,
    test_loader=skin_test_cnn_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=False,
    epochs=CNN_EPOCHS
)

cnn_skin_no_bn_result

In [ ]:
CNN_EPOCHS = 3

cnn_camera_no_bn_model, cnn_camera_no_bn_history, cnn_camera_no_bn_result = run_cnn_experiment(
    experiment_name="CNN camera fara BatchNorm",
    train_loader=camera_train_cnn_loader,
    test_loader=camera_test_cnn_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=False,
    epochs=CNN_EPOCHS
)

cnn_camera_no_bn_result

In [ ]:
def make_cnn_aug_transform(mean, std, image_size=64, mode="none", dataset_type="skin"):
    transforms_list = [
        A.Resize(image_size, image_size)
    ]

    if mode == "geometric":
        if dataset_type == "skin":
            transforms_list += [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Rotate(limit=45, p=0.5),
            ]
        else:
            transforms_list += [
                A.HorizontalFlip(p=0.5),
                A.Affine(
                    scale=(0.90, 1.10),
                    translate_percent=(-0.05, 0.05),
                    rotate=(-12, 12),
                    p=0.7
                ),
            ]

    elif mode == "color":
        transforms_list += [
            A.RandomBrightnessContrast(
                brightness_limit=0.15,
                contrast_limit=0.15,
                p=0.7
            ),
            A.GaussianBlur(blur_limit=(3, 3), p=0.2),
        ]

    elif mode == "all":
        if dataset_type == "skin":
            transforms_list += [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Rotate(limit=45, p=0.5),
                A.RandomBrightnessContrast(
                    brightness_limit=0.10,
                    contrast_limit=0.10,
                    p=0.7
                ),
                A.GaussianBlur(blur_limit=(3, 3), p=0.15),
            ]
        else:
            transforms_list += [
                A.HorizontalFlip(p=0.5),
                A.Affine(
                    scale=(0.90, 1.10),
                    translate_percent=(-0.05, 0.05),
                    rotate=(-12, 12),
                    p=0.7
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=0.20,
                    contrast_limit=0.20,
                    p=0.7
                ),
                A.GaussianBlur(blur_limit=(3, 3), p=0.15),
            ]

    elif mode == "none":
        pass

    else:
        raise ValueError("mode trebuie să fie: none, geometric, color sau all")

    transforms_list += [
        A.Normalize(mean=mean, std=std),
        ToTensorV2()
    ]

    return A.Compose(transforms_list)

In [ ]:
def make_cnn_ablation_loaders(dataset_name: str, aug_mode: str):
    if dataset_name == "skin":
        train_transform = make_cnn_aug_transform(
            mean=SKIN_MEAN,
            std=SKIN_STD,
            image_size=CNN_IMAGE_SIZE,
            mode=aug_mode,
            dataset_type="skin"
        )

        test_transform = make_cnn_aug_transform(
            mean=SKIN_MEAN,
            std=SKIN_STD,
            image_size=CNN_IMAGE_SIZE,
            mode="none",
            dataset_type="skin"
        )

        train_ds = FastImageCSVDataset(
            df=df_mole_train,
            root_dirs=[MOLE_TRAIN_DIR, MOLE_ROOT],
            image_column=MOLE_PATH_COL,
            label_column=MOLE_LABEL_COL,
            class_to_idx=skin_class_to_idx,
            transform=train_transform
        )

        test_ds = FastImageCSVDataset(
            df=df_mole_test,
            root_dirs=[MOLE_TEST_DIR, MOLE_ROOT],
            image_column=MOLE_PATH_COL,
            label_column=MOLE_LABEL_COL,
            class_to_idx=skin_class_to_idx,
            transform=test_transform
        )

    elif dataset_name == "camera":
        train_transform = make_cnn_aug_transform(
            mean=CAMERA_MEAN,
            std=CAMERA_STD,
            image_size=CNN_IMAGE_SIZE,
            mode=aug_mode,
            dataset_type="camera"
        )

        test_transform = make_cnn_aug_transform(
            mean=CAMERA_MEAN,
            std=CAMERA_STD,
            image_size=CNN_IMAGE_SIZE,
            mode="none",
            dataset_type="camera"
        )

        train_ds = FastImageCSVDataset(
            df=df_face_train,
            root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
            image_column=FACE_PATH_COL,
            label_column=FACE_LABEL_COL,
            class_to_idx=camera_class_to_idx,
            transform=train_transform
        )

        test_ds = FastImageCSVDataset(
            df=df_face_test,
            root_dirs=[FACE_DATASET_DIR, FACE_ROOT],
            image_column=FACE_PATH_COL,
            label_column=FACE_LABEL_COL,
            class_to_idx=camera_class_to_idx,
            transform=test_transform
        )

    else:
        raise ValueError("dataset_name trebuie să fie 'skin' sau 'camera'")

    train_loader = DataLoader(
        train_ds,
        batch_size=CNN_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=CNN_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    return train_loader, test_loader

In [ ]:
RUN_SINGLE_AUG_EXPERIMENT = False 

if RUN_SINGLE_AUG_EXPERIMENT:
    DATASET_NAME = "skin"     
    AUG_MODE = "geometric"     
    CNN_EPOCHS = 3

    train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
        dataset_name=DATASET_NAME,
        aug_mode=AUG_MODE
    )

    if DATASET_NAME == "skin":
        num_classes = len(skin_class_to_idx)
        labels = skin_class_labels
        first_kernel = 5
    else:
        num_classes = len(camera_class_to_idx)
        labels = camera_class_labels
        first_kernel = 3

    cnn_aug_model, cnn_aug_history, cnn_aug_result = run_cnn_experiment(
        experiment_name=f"CNN {DATASET_NAME} augmentare {AUG_MODE}",
        train_loader=train_loader_aug,
        test_loader=test_loader_aug,
        num_classes=num_classes,
        class_labels=labels,
        first_kernel_size=first_kernel,
        use_batchnorm=True,
        epochs=CNN_EPOCHS
    )

    display(cnn_aug_result)
else:
    print("Sărit: experimentul punctual este opțional; rulează ablațiile individuale none/geometric/color/all de mai jos.")

In [ ]:
DATASET_NAME = "skin"
AUG_MODE = "none"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_skin_aug_none_model, cnn_skin_aug_none_history, cnn_skin_aug_none_result = run_cnn_experiment(
    experiment_name="CNN skin augmentare none",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_skin_aug_none_result

In [ ]:
DATASET_NAME = "skin"
AUG_MODE = "color"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_skin_aug_color_model, cnn_skin_aug_color_history, cnn_skin_aug_color_result = run_cnn_experiment(
    experiment_name="CNN skin augmentare color",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_skin_aug_color_result


In [ ]:
DATASET_NAME = "skin"
AUG_MODE = "all"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_skin_aug_all_model, cnn_skin_aug_all_history, cnn_skin_aug_all_result = run_cnn_experiment(
    experiment_name="CNN skin augmentare all",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_skin_aug_all_result

In [ ]:
DATASET_NAME = "camera"
AUG_MODE = "none"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_camera_aug_none_model, cnn_camera_aug_none_history, cnn_camera_aug_none_result = run_cnn_experiment(
    experiment_name="CNN camera augmentare none",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_aug_none_result

In [ ]:
DATASET_NAME = "camera"
AUG_MODE = "geometric"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_camera_aug_geometric_model, cnn_camera_aug_geometric_history, cnn_camera_aug_geometric_result = run_cnn_experiment(
    experiment_name="CNN camera augmentare geometric",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_aug_geometric_result

In [ ]:
DATASET_NAME = "camera"
AUG_MODE = "color"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_camera_aug_color_model, cnn_camera_aug_color_history, cnn_camera_aug_color_result = run_cnn_experiment(
    experiment_name="CNN camera augmentare color",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_aug_color_result

In [ ]:
DATASET_NAME = "camera"
AUG_MODE = "all"
CNN_EPOCHS = 3

train_loader_aug, test_loader_aug = make_cnn_ablation_loaders(
    dataset_name=DATASET_NAME,
    aug_mode=AUG_MODE
)

cnn_camera_aug_all_model, cnn_camera_aug_all_history, cnn_camera_aug_all_result = run_cnn_experiment(
    experiment_name="CNN camera augmentare all",
    train_loader=train_loader_aug,
    test_loader=test_loader_aug,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_aug_all_result

In [ ]:
def make_weighted_sampler(dataset):
    labels = np.array([label for _, label in dataset.samples])

    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    print("Class counts:", class_counts)
    print("Class weights:", class_weights)

    return sampler

In [ ]:
skin_sampler = make_weighted_sampler(skin_train_cnn_ds)

skin_train_cnn_balanced_loader = DataLoader(
    skin_train_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    sampler=skin_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

CNN_EPOCHS = 3

cnn_skin_balanced_model, cnn_skin_balanced_history, cnn_skin_balanced_result = run_cnn_experiment(
    experiment_name="CNN skin cu WeightedRandomSampler",
    train_loader=skin_train_cnn_balanced_loader,
    test_loader=skin_test_cnn_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_skin_balanced_result

In [ ]:
camera_sampler = make_weighted_sampler(camera_train_cnn_ds)

camera_train_cnn_balanced_loader = DataLoader(
    camera_train_cnn_ds,
    batch_size=CNN_BATCH_SIZE,
    sampler=camera_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

CNN_EPOCHS = 3

cnn_camera_balanced_model, cnn_camera_balanced_history, cnn_camera_balanced_result = run_cnn_experiment(
    experiment_name="CNN camera cu WeightedRandomSampler",
    train_loader=camera_train_cnn_balanced_loader,
    test_loader=camera_test_cnn_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    first_kernel_size=3,
    use_batchnorm=True,
    epochs=CNN_EPOCHS
)

cnn_camera_balanced_result

In [ ]:
def run_finetune_experiment_with_warmup(
    experiment_name: str,
    train_loader,
    test_loader,
    num_classes: int,
    class_labels: List[str],
    epochs: int = 3,
    lr: float = 1e-3,
    warmup_epochs: int = 1,
    weight_decay: float = 1e-4,
    freeze_backbone: bool = True
):
    print(f"\n=== Experiment fine-tuning cu warmup: {experiment_name} ===")

    model = build_mobilenet_v2(
        num_classes=num_classes,
        freeze_backbone=freeze_backbone
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    def warmup_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        return 1.0

    scheduler = LambdaLR(optimizer, lr_lambda=warmup_lambda)

    history_rows = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch_finetune(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            freeze_bn=freeze_backbone
        )

        val_metrics = evaluate_finetune(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=DEVICE
        )

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_f1_micro": train_metrics["f1_micro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        history_rows.append(row)

        print(
            f"Epoca {epoch:02d}/{epochs} | "
            f"train F1={row['train_f1_macro']:.4f} | "
            f"test F1={row['val_f1_macro']:.4f} | "
            f"lr={row['lr']:.6f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_finetune(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE
    )

    history = pd.DataFrame(history_rows)
    safe_name = experiment_name.replace(" ", "_").replace("/", "_").lower()

    history.to_csv(ARTIFACTS_DIR / f"{safe_name}_history.csv", index=False)

    plot_training_curves(
        history=history,
        title=f"Curbe train/test — {experiment_name}",
        output_name=f"{safe_name}_curves.png"
    )

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=class_labels,
        title=f"Matrice de confuzie — {experiment_name}",
        output_name=f"{safe_name}_confusion_matrix.png"
    )

    result = {
        "experiment": experiment_name,
        "model": "MobileNetV2 pretrained ImageNet",
        "freeze_backbone": freeze_backbone,
        "warmup": True,
        "warmup_epochs": warmup_epochs,
        "best_val_f1_macro": best_f1,
        "final_accuracy": final_metrics["accuracy"],
        "final_f1_macro": final_metrics["f1_macro"],
        "final_f1_micro": final_metrics["f1_micro"],
        "final_loss": final_metrics["loss"],
    }

    with open(ARTIFACTS_DIR / f"{safe_name}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nRezultat final:")
    print(result)

    return model, history, result

In [ ]:
FINETUNE_EPOCHS = 3

finetune_skin_warmup_model, finetune_skin_warmup_history, finetune_skin_warmup_result = run_finetune_experiment_with_warmup(
    experiment_name="MobileNetV2 skin freeze backbone cu warmup",
    train_loader=skin_train_ft_loader,
    test_loader=skin_test_ft_loader,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    epochs=FINETUNE_EPOCHS,
    lr=1e-3,
    warmup_epochs=1,
    freeze_backbone=True
)

finetune_skin_warmup_result

In [ ]:
FINETUNE_EPOCHS = 3

finetune_camera_warmup_model, finetune_camera_warmup_history, finetune_camera_warmup_result = run_finetune_experiment_with_warmup(
    experiment_name="MobileNetV2 camera freeze backbone cu warmup",
    train_loader=camera_train_ft_loader,
    test_loader=camera_test_ft_loader,
    num_classes=len(camera_class_to_idx),
    class_labels=camera_class_labels,
    epochs=FINETUNE_EPOCHS,
    lr=1e-3,
    warmup_epochs=1,
    freeze_backbone=True
)

finetune_camera_warmup_result

In [ ]:
def load_all_experiment_results():
    rows = []

    result_files = list(ARTIFACTS_DIR.glob("*_results.json")) + list(ARTIFACTS_DIR.glob("*_result.json"))

    for path in sorted(result_files):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            rows.append({
                "experiment": data.get("experiment", data.get("run_name", path.stem)),
                "model": data.get("model", ""),
                "best_val_f1_macro": data.get("best_val_f1_macro"),
                "final_accuracy": data.get("final_accuracy"),
                "final_f1_macro": data.get("final_f1_macro"),
                "final_f1_micro": data.get("final_f1_micro"),
                "final_loss": data.get("final_loss"),
                "file": path.name,
            })

        except Exception as e:
            print("Nu pot citi:", path, e)

    columns = [
        "experiment",
        "model",
        "best_val_f1_macro",
        "final_accuracy",
        "final_f1_macro",
        "final_f1_micro",
        "final_loss",
        "file",
    ]

    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns).sort_values("experiment")


results_35_df = load_all_experiment_results()
display(results_35_df)

results_35_df.to_csv(
    ARTIFACTS_DIR / "tabel_final_ablatii_3_5.csv",
    index=False
)

print("Tabel salvat în:", ARTIFACTS_DIR / "tabel_final_ablatii_3_5.csv")

In [ ]:

SKIN_BOOST_IMAGE_SIZE = 224
SKIN_BOOST_BATCH_SIZE = 32
SKIN_BOOST_EPOCHS = 12
SKIN_BOOST_HEAD_EPOCHS = 2
SKIN_BOOST_LR_BACKBONE = 1e-4
SKIN_BOOST_LR_HEAD = 8e-4
SKIN_BOOST_WEIGHT_DECAY = 1e-4


def make_skin_boost_transforms(image_size=224):
    train_tf = A.Compose([
        A.Resize(256, 256),
        A.RandomCrop(image_size, image_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            scale=(0.85, 1.15),
            translate_percent=(-0.05, 0.05),
            rotate=(-25, 25),
            shear=(-5, 5),
            p=0.60,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=0.60,
        ),
        A.HueSaturationValue(
            hue_shift_limit=8,
            sat_shift_limit=12,
            val_shift_limit=8,
            p=0.35,
        ),
        A.GaussianBlur(blur_limit=(3, 3), p=0.10),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

    test_tf = A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

    return train_tf, test_tf


skin_boost_train_transform, skin_boost_test_transform = make_skin_boost_transforms(SKIN_BOOST_IMAGE_SIZE)

skin_train_boost_ds = FastImageCSVDataset(
    df=df_mole_train,
    root_dirs=[MOLE_TRAIN_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_boost_train_transform,
)

skin_test_boost_ds = FastImageCSVDataset(
    df=df_mole_test,
    root_dirs=[MOLE_TEST_DIR, MOLE_ROOT],
    image_column=MOLE_PATH_COL,
    label_column=MOLE_LABEL_COL,
    class_to_idx=skin_class_to_idx,
    transform=skin_boost_test_transform,
)


def get_labels_from_dataset(dataset):
    return np.array([label for _, label in dataset.samples], dtype=np.int64)


def make_balanced_sampler_and_weights(dataset, beta=0.999):
    labels = get_labels_from_dataset(dataset)
    counts = np.bincount(labels, minlength=len(skin_class_to_idx)).astype(np.float32)

    # Sampler moderat: inverse sqrt, nu inverse complet, ca să nu supraînvețe clasele foarte rare.
    sample_class_weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))
    sample_weights = sample_class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

    # Effective number of samples: class weights pentru loss, mai stabil decât 1/count brut.
    effective_num = 1.0 - np.power(beta, counts)
    class_weights = (1.0 - beta) / np.maximum(effective_num, 1e-8)
    class_weights = class_weights / class_weights.sum() * len(counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    print("Distribuție clase skin:")
    for idx, cnt in enumerate(counts.astype(int)):
        print(f"  {idx} ({skin_idx_to_class[idx]}): {cnt}")
    print("Class weights loss:", class_weights.numpy().round(3))

    return sampler, class_weights


skin_boost_sampler, skin_boost_class_weights = make_balanced_sampler_and_weights(skin_train_boost_ds)

skin_train_boost_loader = DataLoader(
    skin_train_boost_ds,
    batch_size=SKIN_BOOST_BATCH_SIZE,
    sampler=skin_boost_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

skin_test_boost_loader = DataLoader(
    skin_test_boost_ds,
    batch_size=SKIN_BOOST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)


def build_strong_skin_model(num_classes: int):
    """Preferă EfficientNet-B0 pretrained; fallback MobileNetV2. Nu îngheață random weights."""
    # 1) EfficientNet-B0, de obicei mai bun pe imagini medicale mici decât MobileNetV2.
    try:
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(in_features, num_classes),
        )
        print("Model boost: EfficientNet-B0 cu weights ImageNet.")
        return model, True, "EfficientNet-B0"
    except Exception as exc_eff:
        print("Nu am putut încărca EfficientNet-B0 pretrained:", repr(exc_eff))

    # 2) MobileNetV2 pretrained.
    try:
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(in_features, num_classes),
        )
        print("Model boost: MobileNetV2 cu weights ImageNet.")
        return model, True, "MobileNetV2"
    except Exception as exc_mob:
        print("Nu am putut încărca MobileNetV2 pretrained:", repr(exc_mob))

    # 3) Fallback offline: MobileNetV2 random, dar antrenăm tot backbone-ul.
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_features, num_classes),
    )
    print("Model boost: MobileNetV2 fără pretrained weights. Antrenez tot modelul, NU îngheț backbone-ul.")
    return model, False, "MobileNetV2-random"


def set_backbone_trainable(model, trainable: bool):
    for name, param in model.named_parameters():
        if name.startswith("classifier"):
            param.requires_grad = True
        else:
            param.requires_grad = trainable


def make_optimizer_for_strong_skin(model, pretrained_loaded: bool, head_only: bool):
    if head_only:
        params = [p for p in model.parameters() if p.requires_grad]
        return optim.AdamW(params, lr=2e-3, weight_decay=SKIN_BOOST_WEIGHT_DECAY)

    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("classifier"):
            head_params.append(param)
        else:
            backbone_params.append(param)

    if pretrained_loaded:
        return optim.AdamW(
            [
                {"params": backbone_params, "lr": SKIN_BOOST_LR_BACKBONE},
                {"params": head_params, "lr": SKIN_BOOST_LR_HEAD},
            ],
            weight_decay=SKIN_BOOST_WEIGHT_DECAY,
        )
    else:
        return optim.AdamW(model.parameters(), lr=5e-4, weight_decay=SKIN_BOOST_WEIGHT_DECAY)


def train_one_epoch_strong(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp_autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        if USE_AMP:
            AMP_SCALER.scale(loss).backward()
            AMP_SCALER.step(optimizer)
            AMP_SCALER.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
    }


@torch.no_grad()
def evaluate_strong(model, loader, criterion, device, use_tta=False):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp_autocast():
            logits = model(images)
            if use_tta:
                
                logits = (
                    logits
                    + model(torch.flip(images, dims=[3]))
                    + model(torch.flip(images, dims=[2]))
                    + model(torch.rot90(images, k=1, dims=[2, 3]))
                ) / 4.0
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
    }


def run_strong_skin_training():
    global strong_skin_model, strong_skin_history, strong_skin_result

    model, pretrained_loaded, model_name = build_strong_skin_model(num_classes=len(skin_class_to_idx))
    model = model.to(DEVICE)

    class_weights = skin_boost_class_weights.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    best_f1 = -1.0
    best_state = None
    rows = []

    
    if pretrained_loaded and SKIN_BOOST_HEAD_EPOCHS > 0:
        set_backbone_trainable(model, trainable=False)
        optimizer = make_optimizer_for_strong_skin(model, pretrained_loaded=True, head_only=True)

        for epoch in range(1, SKIN_BOOST_HEAD_EPOCHS + 1):
            train_metrics = train_one_epoch_strong(model, skin_train_boost_loader, criterion, optimizer, DEVICE)
            val_metrics = evaluate_strong(model, skin_test_boost_loader, criterion, DEVICE, use_tta=False)

            row = {
                "phase": "head_only",
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_f1_macro": train_metrics["f1_macro"],
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_f1_macro": val_metrics["f1_macro"],
                "val_f1_micro": val_metrics["f1_micro"],
                "lr": optimizer.param_groups[0]["lr"],
            }
            rows.append(row)
            print(
                f"Head epoca {epoch:02d}/{SKIN_BOOST_HEAD_EPOCHS} | "
                f"train F1={row['train_f1_macro']:.4f} | val F1={row['val_f1_macro']:.4f}"
            )

            if val_metrics["f1_macro"] > best_f1:
                best_f1 = val_metrics["f1_macro"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


    set_backbone_trainable(model, trainable=True)
    optimizer = make_optimizer_for_strong_skin(model, pretrained_loaded=pretrained_loaded, head_only=False)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, SKIN_BOOST_EPOCHS), eta_min=1e-6)

    for epoch in range(1, SKIN_BOOST_EPOCHS + 1):
        train_metrics = train_one_epoch_strong(model, skin_train_boost_loader, criterion, optimizer, DEVICE)
        val_metrics = evaluate_strong(model, skin_test_boost_loader, criterion, DEVICE, use_tta=False)
        scheduler.step()

        row = {
            "phase": "full_finetune",
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_f1_macro": train_metrics["f1_macro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        rows.append(row)
        print(
            f"Full epoca {epoch:02d}/{SKIN_BOOST_EPOCHS} | "
            f"train F1={row['train_f1_macro']:.4f} | val F1={row['val_f1_macro']:.4f} | lr={row['lr']:.6f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_strong(model, skin_test_boost_loader, criterion, DEVICE, use_tta=True)
    history = pd.DataFrame(rows)

    history.to_csv(ARTIFACTS_DIR / "strong_skin_history.csv", index=False)
    torch.save(model.state_dict(), CHECKPOINT_DIR / "strong_skin_model.pt")

    result = {
        "experiment": "Strong skin fine-tuning pentru Kaggle",
        "model": model_name,
        "pretrained_loaded": pretrained_loaded,
        "image_size": SKIN_BOOST_IMAGE_SIZE,
        "batch_size": SKIN_BOOST_BATCH_SIZE,
        "best_val_f1_macro_no_tta": float(best_f1),
        "final_val_f1_macro_tta": float(final_metrics["f1_macro"]),
        "final_accuracy_tta": float(final_metrics["accuracy"]),
        "final_f1_micro_tta": float(final_metrics["f1_micro"]),
        "final_loss_tta": float(final_metrics["loss"]),
    }

    with open(ARTIFACTS_DIR / "strong_skin_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nRezultat strong skin:")
    print(result)

    try:
        plot_training_curves(
            history=history.rename(columns={"val_f1_macro": "val_f1_macro"}),
            title="Curbe train/test — Strong skin fine-tuning",
            output_name="strong_skin_curves.png",
        )
    except Exception as exc:
        print("Nu am putut desena curbele strong skin:", repr(exc))

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=skin_class_labels,
        title="Matrice de confuzie — Strong skin fine-tuning + TTA",
        output_name="strong_skin_confusion_matrix.png",
    )

    strong_skin_model = model
    strong_skin_history = history
    strong_skin_result = result

    return model, history, result


# Rulează antrenarea boost pentru skin. Dacă vrei test rapid, setează SKIN_BOOST_EPOCHS = 3 mai sus.
strong_skin_model, strong_skin_history, strong_skin_result = run_strong_skin_training()
strong_skin_result


## Submission Kaggle



In [ ]:
class FastImageCSVTestDataset(Dataset):
    def __init__(self, df, root_dirs, image_column, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dirs = [Path(r) for r in root_dirs if r is not None and Path(r).exists()]
        self.image_column = image_column
        self.transform = transform
        self.image_index = build_image_index_fast(self.root_dirs)
        self.samples = []

        for _, row in self.df.iterrows():
            value = str(row[self.image_column])
            img_path = resolve_image_path_fast(value, self.root_dirs, self.image_index)
            self.samples.append((img_path, value))

        print(f"Test dataset creat: {len(self.samples)} imagini")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, sample_id = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, sample_id


def pick_first_existing_model(candidates):
    """candidates = [(nume_variabilă_model, transform)]"""
    for model_name, transform in candidates:
        model = globals().get(model_name)
        if isinstance(model, nn.Module):
            print(f"Folosesc modelul pentru submission: {model_name}")
            return model, transform, model_name
    raise RuntimeError("Nu există niciun model antrenat în memorie. Rulează întâi celulele de CNN/fine-tuning.")


@torch.no_grad()
def generate_predictions(model, loader, device, use_tta=False):
    model.eval()
    all_ids, all_preds = [], []
    for images, sample_ids in loader:
        images = images.to(device, non_blocking=True)
        with amp_autocast():
            logits = model(images)
            if use_tta:
                
                logits = (
                    logits
                    + model(torch.flip(images, dims=[3]))
                    + model(torch.flip(images, dims=[2]))
                    + model(torch.rot90(images, k=1, dims=[2, 3]))
                ) / 4.0
        preds = logits.argmax(dim=1).detach().cpu().numpy().tolist()
        all_ids.extend(list(sample_ids))
        all_preds.extend(preds)
    return all_ids, all_preds


def detect_test_image_column(df: pd.DataFrame):
    for col in ["imagine", "image", "path", "filename", "file", "id"]:
        if col in df.columns:
            return col
    return df.columns[0]


def save_kaggle_submissions():
    generated = []

    
    if MOLE_KAGGLE_TEST_CSV is not None and MOLE_KAGGLE_TEST_DIR is not None:
        df_mole_remote = pd.read_csv(MOLE_KAGGLE_TEST_CSV)
        mole_id_col = detect_test_image_column(df_mole_remote)
        skin_model, skin_transform, skin_model_name = pick_first_existing_model([
            ("strong_skin_model", globals().get("skin_boost_test_transform", finetune_transform)),
            ("finetune_skin_warmup_model", finetune_transform),
            ("finetune_skin_model", finetune_transform),
            ("cnn_skin_balanced_model", skin_cnn_transform),
            ("cnn_skin_model", skin_cnn_transform),
        ])

        mole_remote_ds = FastImageCSVTestDataset(
            df=df_mole_remote,
            root_dirs=[MOLE_KAGGLE_TEST_DIR, MOLE_ROOT],
            image_column=mole_id_col,
            transform=skin_transform,
        )
        mole_remote_loader = DataLoader(
            mole_remote_ds,
            batch_size=SKIN_BOOST_BATCH_SIZE if skin_model_name == "strong_skin_model" else (FINETUNE_BATCH_SIZE if skin_transform is finetune_transform else CNN_BATCH_SIZE),
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        use_skin_tta = skin_model_name == "strong_skin_model"
        print("TTA pentru mole:", use_skin_tta)
        ids_mole, preds_mole = generate_predictions(skin_model, mole_remote_loader, DEVICE, use_tta=use_skin_tta)
        preds_mole_labels = [skin_idx_to_class[int(p)] for p in preds_mole]

        # Format numeric compatibil cu multe competiții Kaggle.
        df_submission_mole = pd.DataFrame({"Id": ids_mole, "Prediction": preds_mole})
        mole_numeric_path = WORK_DIR / "submission_mole.csv"
        df_submission_mole.to_csv(mole_numeric_path, index=False)
        generated.append(mole_numeric_path)

        # Variantă cu etichete text, utilă dacă platforma cere diagnosticul ca string.
        df_submission_mole_labels = pd.DataFrame({"Id": ids_mole, "diagnostic": preds_mole_labels})
        mole_labels_path = WORK_DIR / "submission_mole_labels.csv"
        df_submission_mole_labels.to_csv(mole_labels_path, index=False)
        generated.append(mole_labels_path)

        print("Mole submission salvat:", mole_numeric_path)
        display(df_submission_mole.head())
        print(df_submission_mole["Prediction"].value_counts())
    else:
        print("Nu am găsit kaggle_test pentru mole; sar peste submission_mole.csv")

    
    if FACE_REMOTE_TEST_CSV is not None:
        df_face_remote_local = pd.read_csv(FACE_REMOTE_TEST_CSV)
        face_id_col = detect_test_image_column(df_face_remote_local)
        face_model, face_transform, face_model_name = pick_first_existing_model([
            ("finetune_camera_warmup_model", finetune_transform),
            ("finetune_camera_model", finetune_transform),
            ("cnn_camera_balanced_model", camera_cnn_transform),
            ("cnn_camera_model", camera_cnn_transform),
        ])

        face_remote_ds = FastImageCSVTestDataset(
            df=df_face_remote_local,
            root_dirs=[FACE_DATASET_DIR / "test", FACE_DATASET_DIR, FACE_ROOT],
            image_column=face_id_col,
            transform=face_transform,
        )
        face_remote_loader = DataLoader(
            face_remote_ds,
            batch_size=FINETUNE_BATCH_SIZE if face_transform is finetune_transform else CNN_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        ids_face, preds_face = generate_predictions(face_model, face_remote_loader, DEVICE)
        preds_face_labels = [int(camera_idx_to_class[int(p)]) for p in preds_face]

        df_submission_face = pd.DataFrame({"id": ids_face, "label": preds_face_labels})
        face_path = WORK_DIR / "submission_face.csv"
        df_submission_face.to_csv(face_path, index=False)
        generated.append(face_path)

        print("Face submission salvat:", face_path)
        display(df_submission_face.head())
        print(df_submission_face["label"].value_counts().sort_index())
    else:
        print("Nu am găsit remote_test.csv pentru face; sar peste submission_face.csv")

    print("\nFișiere generate:")
    for p in generated:
        print(" -", p)
    return generated


generated_submission_files = save_kaggle_submissions()


In [ ]:
import copy

def compute_image_difficulty_scores(dataset, max_size=32):
    """
    Calculează un scor simplu de dificultate pentru fiecare imagine.

    Idee:
    - imagini cu contrast mic sunt considerate mai grele;
    - imagini foarte întunecate sau foarte luminoase sunt considerate mai grele;
    - scor mic = exemplu ușor;
    - scor mare = exemplu greu.
    """
    scores = []

    for img_path, label in dataset.samples:
        img = Image.open(img_path).convert("L")
        img = img.resize((max_size, max_size))

        arr = np.asarray(img).astype(np.float32) / 255.0

        brightness = arr.mean()
        contrast = arr.std()

        brightness_penalty = abs(brightness - 0.5)
        contrast_penalty = 1.0 - contrast

        difficulty = brightness_penalty + contrast_penalty
        scores.append(difficulty)

    scores = np.array(scores)

    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    return scores


skin_difficulty_scores = compute_image_difficulty_scores(skin_train_cnn_ds)

print("Difficulty scores:")
print("min:", skin_difficulty_scores.min())
print("max:", skin_difficulty_scores.max())
print("mean:", skin_difficulty_scores.mean())

In [ ]:
def pacing_function(epoch, total_epochs, start_fraction=0.35):
    """
    În prima epocă folosim doar o parte din exemplele cele mai ușoare.
    Treptat, includem și exemple mai grele.
    """
    progress = (epoch - 1) / max(1, total_epochs - 1)
    fraction = start_fraction + (1.0 - start_fraction) * progress
    return min(1.0, fraction)

In [ ]:

def run_cnn_curriculum_experiment(
    experiment_name: str,
    train_dataset,
    test_loader,
    difficulty_scores,
    num_classes: int,
    class_labels: List[str],
    first_kernel_size: int = 5,
    use_batchnorm: bool = True,
    epochs: int = 4,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    start_fraction: float = 0.35
):
    print(f"\n=== Experiment Curriculum Learning: {experiment_name} ===")

    sorted_indices = np.argsort(difficulty_scores)

    model = SimpleCNNClassifier(
        num_classes=num_classes,
        first_kernel_size=first_kernel_size,
        use_batchnorm=use_batchnorm,
        dropout_p=0.25
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )

    history_rows = []
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        fraction = pacing_function(
            epoch=epoch,
            total_epochs=epochs,
            start_fraction=start_fraction
        )

        n_used = int(len(sorted_indices) * fraction)
        current_indices = sorted_indices[:n_used]

        current_subset = Subset(train_dataset, current_indices)

        current_loader = DataLoader(
            current_subset,
            batch_size=CNN_BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            drop_last=True
        )

        train_metrics = train_one_epoch(
            model=model,
            loader=current_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE
        )

        val_metrics = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=DEVICE
        )

        scheduler.step(val_metrics["f1_macro"])

        row = {
            "epoch": epoch,
            "used_fraction": fraction,
            "used_samples": n_used,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "train_f1_micro": train_metrics["f1_micro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_micro": val_metrics["f1_micro"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        history_rows.append(row)

        print(
            f"Epoca {epoch:02d}/{epochs} | "
            f"date folosite={n_used}/{len(train_dataset)} ({fraction:.2f}) | "
            f"train F1={row['train_f1_macro']:.4f} | "
            f"test F1={row['val_f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE
    )

    history = pd.DataFrame(history_rows)
    safe_name = experiment_name.replace(" ", "_").replace("/", "_").lower()

    history.to_csv(
        ARTIFACTS_DIR / f"{safe_name}_history.csv",
        index=False
    )

    plot_training_curves(
        history=history,
        title=f"Curbe train/test — {experiment_name}",
        output_name=f"{safe_name}_curves.png"
    )

    plot_confusion_matrix_simple(
        y_true=final_metrics["labels"],
        y_pred=final_metrics["preds"],
        class_labels=class_labels,
        title=f"Matrice de confuzie — {experiment_name}",
        output_name=f"{safe_name}_confusion_matrix.png"
    )

    result = {
        "experiment": experiment_name,
        "model": "CNN curriculum learning",
        "curriculum": True,
        "start_fraction": start_fraction,
        "best_val_f1_macro": best_f1,
        "final_accuracy": final_metrics["accuracy"],
        "final_f1_macro": final_metrics["f1_macro"],
        "final_f1_micro": final_metrics["f1_micro"],
        "final_loss": final_metrics["loss"],
    }

    with open(ARTIFACTS_DIR / f"{safe_name}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nRezultat final:")
    print(result)

    return model, history, result

In [ ]:
CURRICULUM_EPOCHS = 4

cnn_skin_curriculum_model, cnn_skin_curriculum_history, cnn_skin_curriculum_result = run_cnn_curriculum_experiment(
    experiment_name="CNN skin curriculum learning",
    train_dataset=skin_train_cnn_ds,
    test_loader=skin_test_cnn_loader,
    difficulty_scores=skin_difficulty_scores,
    num_classes=len(skin_class_to_idx),
    class_labels=skin_class_labels,
    first_kernel_size=5,
    use_batchnorm=True,
    epochs=CURRICULUM_EPOCHS,
    lr=1e-3,
    start_fraction=0.35
)

cnn_skin_curriculum_result

In [ ]:
def load_all_experiment_results():
    rows = []

    result_files = list(ARTIFACTS_DIR.glob("*_results.json")) + list(ARTIFACTS_DIR.glob("*_result.json"))

    for path in sorted(result_files):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            rows.append({
                "experiment": data.get("experiment", data.get("run_name", path.stem)),
                "model": data.get("model", ""),
                "best_val_f1_macro": data.get("best_val_f1_macro"),
                "final_accuracy": data.get("final_accuracy"),
                "final_f1_macro": data.get("final_f1_macro"),
                "final_f1_micro": data.get("final_f1_micro"),
                "final_loss": data.get("final_loss"),
                "file": path.name,
            })

        except Exception as e:
            print("Nu pot citi:", path, e)

    columns = [
        "experiment",
        "model",
        "best_val_f1_macro",
        "final_accuracy",
        "final_f1_macro",
        "final_f1_micro",
        "final_loss",
        "file",
    ]

    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns).sort_values("experiment")


results_35_df = load_all_experiment_results()
display(results_35_df)

results_35_df.to_csv(
    ARTIFACTS_DIR / "tabel_final_ablatii_3_5.csv",
    index=False
)

print("Tabel salvat în:", ARTIFACTS_DIR / "tabel_final_ablatii_3_5.csv")

In [ ]:
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display, FileLink

SEED = 7319
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")

WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)


def find_camera_csvs():
    best = None

    for train_csv in INPUT_DIR.rglob("train.csv"):
        try:
            preview = pd.read_csv(train_csv, nrows=80)
        except Exception:
            continue

        if len(preview.columns) < 2:
            continue

        label_col = preview.columns[1]
        numeric_labels = pd.to_numeric(preview[label_col], errors="coerce")

        score = 0

        # Camera are labeluri 1..7
        if numeric_labels.notna().mean() > 0.8:
            valid_ratio = numeric_labels.dropna().between(1, 7).mean()
            if valid_ratio > 0.8:
                score += 30

        if (train_csv.parent / "local_test.csv").exists():
            score += 10

        if (train_csv.parent / "remote_test.csv").exists():
            score += 10

        path_text = str(train_csv).lower()
        if "camera" in path_text or "candid" in path_text or "face" in path_text:
            score += 5

        if best is None or score > best[0]:
            best = (score, train_csv)

    if best is None:
        raise FileNotFoundError("Nu am găsit train.csv pentru competiția cu camera.")

    train_csv = best[1]
    split_dir = train_csv.parent

    local_test_csv = split_dir / "local_test.csv"
    remote_test_csv = split_dir / "remote_test.csv"

    if not local_test_csv.exists():
        raise FileNotFoundError("Nu am găsit local_test.csv lângă train.csv.")

    if not remote_test_csv.exists():
        raise FileNotFoundError("Nu am găsit remote_test.csv lângă train.csv.")

    return train_csv, local_test_csv, remote_test_csv


FACE_TRAIN_CSV, FACE_LOCAL_TEST_CSV, FACE_REMOTE_TEST_CSV = find_camera_csvs()

FACE_BASE_DIR = FACE_TRAIN_CSV.parent.parent

print("Train CSV:", FACE_TRAIN_CSV)
print("Local test CSV:", FACE_LOCAL_TEST_CSV)
print("Remote test CSV:", FACE_REMOTE_TEST_CSV)
print("Base dir:", FACE_BASE_DIR)

df_face_train = pd.read_csv(FACE_TRAIN_CSV)
df_face_local = pd.read_csv(FACE_LOCAL_TEST_CSV)
df_face_remote = pd.read_csv(FACE_REMOTE_TEST_CSV)

display(df_face_train.head())
display(df_face_local.head())
display(df_face_remote.head())


def guess_path_col(df, label_col=None):
    names = list(df.columns)

    preferred = ["path", "image", "imagine", "filename", "file", "id"]
    for key in preferred:
        for col in names:
            if key in str(col).lower() and col != label_col:
                return col

    return names[0]


def guess_label_col(df):
    names = list(df.columns)

    preferred = ["label", "emotion", "class", "target"]
    for key in preferred:
        for col in names:
            if key in str(col).lower():
                return col

    return names[1]


FACE_LABEL_COL = guess_label_col(df_face_train)
FACE_PATH_COL = guess_path_col(df_face_train, label_col=FACE_LABEL_COL)
FACE_REMOTE_ID_COL = guess_path_col(df_face_remote)

print("FACE_PATH_COL:", FACE_PATH_COL)
print("FACE_LABEL_COL:", FACE_LABEL_COL)
print("FACE_REMOTE_ID_COL:", FACE_REMOTE_ID_COL)

FACE_ORIGINAL_LABELS = sorted(pd.to_numeric(df_face_train[FACE_LABEL_COL]).astype(int).unique().tolist())
FACE_LABEL_TO_IDX = {label: idx for idx, label in enumerate(FACE_ORIGINAL_LABELS)}
FACE_IDX_TO_LABEL = {idx: label for label, idx in FACE_LABEL_TO_IDX.items()}

print("Mapare label original -> index model:", FACE_LABEL_TO_IDX)
print("Mapare index model -> label original:", FACE_IDX_TO_LABEL)

In [ ]:
FACE_IMAGE_SIZE = 224
FACE_BATCH_SIZE = 32       # dacă dă out of memory, schimbă în 16
FACE_NUM_WORKERS = 2
FACE_PIN_MEMORY = torch.cuda.is_available()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

face_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(FACE_IMAGE_SIZE, scale=(0.82, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.16,
        contrast=0.16,
        saturation=0.10,
        hue=0.03,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

face_test_transform = transforms.Compose([
    transforms.Resize((FACE_IMAGE_SIZE, FACE_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


FACE_ROOT_DIRS = [
    FACE_BASE_DIR / "DATASET" / "train",
    FACE_BASE_DIR / "DATASET" / "test",
    FACE_BASE_DIR / "DATASET",
    FACE_BASE_DIR,
    INPUT_DIR,
]

FACE_ROOT_DIRS = [Path(p) for p in FACE_ROOT_DIRS if Path(p).exists()]
print("Root dirs folosite:")
for p in FACE_ROOT_DIRS:
    print(" -", p)


def build_image_index(root_dirs):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    index = {}

    for root in root_dirs:
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in exts:
                index.setdefault(path.name, path)

                try:
                    rel = path.relative_to(root).as_posix()
                    index.setdefault(rel, path)
                except Exception:
                    pass

    return index


FACE_IMAGE_INDEX = build_image_index(FACE_ROOT_DIRS)
print("Imagini indexate:", len(FACE_IMAGE_INDEX))


def resolve_image_path(value):
    value = str(value).strip().replace("\\", "/")
    variants = [value, value.lstrip("./")]

    p = Path(value)
    if p.suffix == "":
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            variants.append(value + ext)
            variants.append(Path(value).name + ext)

    for variant in variants:
        pp = Path(variant)

        if pp.is_absolute() and pp.exists():
            return pp

        for root in FACE_ROOT_DIRS:
            cand1 = root / variant
            if cand1.exists():
                return cand1

            cand2 = root / Path(variant).name
            if cand2.exists():
                return cand2

        if variant in FACE_IMAGE_INDEX:
            return FACE_IMAGE_INDEX[variant]

        base = Path(variant).name
        if base in FACE_IMAGE_INDEX:
            return FACE_IMAGE_INDEX[base]

    raise FileNotFoundError(f"Nu găsesc imaginea pentru valoarea: {value}")


class FaceTrainDataset(Dataset):
    def __init__(self, df, path_col, label_col, transform=None):
        self.df = df.reset_index(drop=True)
        self.path_col = path_col
        self.label_col = label_col
        self.transform = transform

        self.samples = []

        for _, row in self.df.iterrows():
            img_path = resolve_image_path(row[self.path_col])
            original_label = int(row[self.label_col])
            label_idx = FACE_LABEL_TO_IDX[original_label]
            self.samples.append((img_path, label_idx))

        print(f"Dataset cu etichete creat: {len(self.samples)} imagini")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


class FaceRemoteDataset(Dataset):
    def __init__(self, df, id_col, transform=None):
        self.df = df.reset_index(drop=True)
        self.id_col = id_col
        self.transform = transform

        self.samples = []

        for _, row in self.df.iterrows():
            sample_id = str(row[self.id_col])
            img_path = resolve_image_path(sample_id)
            self.samples.append((img_path, sample_id))

        print(f"Remote test creat: {len(self.samples)} imagini")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, sample_id = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, sample_id


face_train_ds = FaceTrainDataset(
    df_face_train,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=face_train_transform,
)

face_local_ds = FaceTrainDataset(
    df_face_local,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=face_test_transform,
)


def make_sampler_and_weights(dataset):
    labels = np.array([label for _, label in dataset.samples], dtype=np.int64)
    counts = np.bincount(labels, minlength=len(FACE_ORIGINAL_LABELS)).astype(np.float32)

    print("Distribuție clase train:")
    for idx, cnt in enumerate(counts.astype(int)):
        print(f"  label original {FACE_IDX_TO_LABEL[idx]}: {cnt}")

    sample_class_weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))
    sample_weights = sample_class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

    beta = 0.995
    effective_num = 1.0 - np.power(beta, counts)
    class_weights = (1.0 - beta) / np.maximum(effective_num, 1e-8)
    class_weights = class_weights / class_weights.mean()
    class_weights = np.clip(class_weights, 0.55, 3.25)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    print("Class weights:", class_weights.numpy().round(3))

    return sampler, class_weights


face_sampler, face_class_weights = make_sampler_and_weights(face_train_ds)

face_train_loader = DataLoader(
    face_train_ds,
    batch_size=FACE_BATCH_SIZE,
    sampler=face_sampler,
    num_workers=FACE_NUM_WORKERS,
    pin_memory=FACE_PIN_MEMORY,
    drop_last=True,
)

face_local_loader = DataLoader(
    face_local_ds,
    batch_size=FACE_BATCH_SIZE,
    shuffle=False,
    num_workers=FACE_NUM_WORKERS,
    pin_memory=FACE_PIN_MEMORY,
)

In [ ]:
FACE_HEAD_EPOCHS = 2
FACE_FULL_EPOCHS = 8

USE_AMP = torch.cuda.is_available()
SCALER = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def build_face_model(num_classes):
    try:
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.40),
            nn.Linear(in_features, num_classes),
        )
        print("Model folosit: EfficientNet-B0 pretrained")
        return model, True
    except Exception as exc:
        print("EfficientNet nu a mers:", repr(exc))

    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.35),
            nn.Linear(in_features, num_classes),
        )
        print("Model folosit: ResNet18 pretrained")
        return model, True
    except Exception as exc:
        print("ResNet18 pretrained nu a mers:", repr(exc))

    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_features, num_classes),
    )
    print("Fallback: ResNet18 fără pretrained")
    return model, False


def is_head_param(name):
    return name.startswith("classifier") or name.startswith("fc")


def set_backbone_trainable(model, trainable):
    for name, param in model.named_parameters():
        if is_head_param(name):
            param.requires_grad = True
        else:
            param.requires_grad = trainable


def freeze_bn(model):
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            module.eval()
            for p in module.parameters():
                p.requires_grad = False


def make_optimizer(model, pretrained_loaded, head_only):
    if head_only:
        return optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=1.5e-3,
            weight_decay=1e-4,
        )

    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if is_head_param(name):
            head_params.append(param)
        else:
            backbone_params.append(param)

    if pretrained_loaded:
        return optim.AdamW(
            [
                {"params": backbone_params, "lr": 8e-5},
                {"params": head_params, "lr": 7e-4},
            ],
            weight_decay=1e-4,
        )

    return optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)


def train_one_epoch_face(model, loader, criterion, optimizer, freeze_batchnorm=False):
    model.train()

    if freeze_batchnorm:
        freeze_bn(model)

    total_loss = 0.0
    preds_all = []
    labels_all = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        SCALER.scale(loss).backward()
        SCALER.step(optimizer)
        SCALER.update()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels_all, preds_all),
        "f1_macro": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_all, preds_all, average="micro", zero_division=0),
    }


@torch.no_grad()
def evaluate_face(model, loader, criterion, use_tta=False):
    model.eval()

    total_loss = 0.0
    preds_all = []
    labels_all = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)

            if use_tta:
                logits_flip = model(torch.flip(images, dims=[3]))
                logits = (logits + logits_flip) / 2.0

            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels_all, preds_all),
        "f1_macro": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_all, preds_all, average="micro", zero_division=0),
        "labels": np.array(labels_all),
        "preds": np.array(preds_all),
    }


face_model, pretrained_loaded = build_face_model(num_classes=len(FACE_ORIGINAL_LABELS))
face_model = face_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=face_class_weights.to(DEVICE),
    label_smoothing=0.04,
)

best_f1 = -1.0
best_state = None
history = []

# Faza 1: antrenăm doar head-ul
if pretrained_loaded:
    print("\nFaza 1: antrenez doar head-ul")
    set_backbone_trainable(face_model, trainable=False)
    optimizer = make_optimizer(face_model, pretrained_loaded=True, head_only=True)

    for epoch in range(1, FACE_HEAD_EPOCHS + 1):
        train_metrics = train_one_epoch_face(
            face_model,
            face_train_loader,
            criterion,
            optimizer,
            freeze_batchnorm=True,
        )

        val_metrics = evaluate_face(
            face_model,
            face_local_loader,
            criterion,
            use_tta=False,
        )

        print(
            f"HEAD epoca {epoch}/{FACE_HEAD_EPOCHS} | "
            f"train F1={train_metrics['f1_macro']:.4f} | "
            f"local F1={val_metrics['f1_macro']:.4f}"
        )

        history.append({
            "phase": "head",
            "epoch": epoch,
            "train_f1": train_metrics["f1_macro"],
            "local_f1": val_metrics["f1_macro"],
            "local_acc": val_metrics["accuracy"],
        })

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in face_model.state_dict().items()}

# Faza 2: fine-tuning complet
print("\nFaza 2: fine-tuning complet")
set_backbone_trainable(face_model, trainable=True)
optimizer = make_optimizer(face_model, pretrained_loaded=pretrained_loaded, head_only=False)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, FACE_FULL_EPOCHS),
    eta_min=1e-6,
)

for epoch in range(1, FACE_FULL_EPOCHS + 1):
    train_metrics = train_one_epoch_face(
        face_model,
        face_train_loader,
        criterion,
        optimizer,
        freeze_batchnorm=False,
    )

    val_metrics = evaluate_face(
        face_model,
        face_local_loader,
        criterion,
        use_tta=False,
    )

    scheduler.step()

    print(
        f"FULL epoca {epoch}/{FACE_FULL_EPOCHS} | "
        f"train F1={train_metrics['f1_macro']:.4f} | "
        f"local F1={val_metrics['f1_macro']:.4f}"
    )

    history.append({
        "phase": "full",
        "epoch": epoch,
        "train_f1": train_metrics["f1_macro"],
        "local_f1": val_metrics["f1_macro"],
        "local_acc": val_metrics["accuracy"],
    })

    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        best_state = {k: v.detach().cpu().clone() for k, v in face_model.state_dict().items()}

if best_state is not None:
    face_model.load_state_dict(best_state)

final_metrics = evaluate_face(
    face_model,
    face_local_loader,
    criterion,
    use_tta=True,
)

print("\nRezultat final local cu TTA:")
print("Accuracy:", final_metrics["accuracy"])
print("F1-macro:", final_metrics["f1_macro"])
print("F1-micro:", final_metrics["f1_micro"])

pd.DataFrame(history).to_csv(WORK_DIR / "face_training_history.csv", index=False)
torch.save(face_model.state_dict(), WORK_DIR / "best_face_model.pt")

print("Model salvat în:", WORK_DIR / "best_face_model.pt")

In [ ]:
face_remote_ds = FaceRemoteDataset(
    df_face_remote,
    FACE_REMOTE_ID_COL,
    transform=face_test_transform,
)

face_remote_loader = DataLoader(
    face_remote_ds,
    batch_size=FACE_BATCH_SIZE,
    shuffle=False,
    num_workers=FACE_NUM_WORKERS,
    pin_memory=FACE_PIN_MEMORY,
)


@torch.no_grad()
def predict_remote_face(model, loader):
    model.eval()

    all_ids = []
    all_preds = []

    for images, sample_ids in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            logits_flip = model(torch.flip(images, dims=[3]))
            logits = (logits + logits_flip) / 2.0

        preds_idx = logits.argmax(dim=1).detach().cpu().numpy().tolist()

        all_ids.extend(list(sample_ids))
        all_preds.extend(preds_idx)

    return all_ids, all_preds


ids_face, preds_idx_face = predict_remote_face(face_model, face_remote_loader)

preds_original_labels = [int(FACE_IDX_TO_LABEL[int(p)]) for p in preds_idx_face]

submission_face = pd.DataFrame({
    "id": ids_face,
    "label": preds_original_labels,
})

submission_face_path = WORK_DIR / "submission_face.csv"
submission_face.to_csv(submission_face_path, index=False)


submission_face_alt = pd.DataFrame({
    "Id": ids_face,
    "Prediction": preds_original_labels,
})

submission_face_alt_path = WORK_DIR / "submission_face_prediction.csv"
submission_face_alt.to_csv(submission_face_alt_path, index=False)

zip_path = WORK_DIR / "face_submissions.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(submission_face_path, arcname="submission_face.csv")
    z.write(submission_face_alt_path, arcname="submission_face_prediction.csv")
    z.write(WORK_DIR / "face_training_history.csv", arcname="face_training_history.csv")

print("Fișier principal pentru competiție:")
print(submission_face_path)

print("\nVariantă alternativă:")
print(submission_face_alt_path)

print("\nPreview:")
display(submission_face.head())

print("\nDistribuție predicții:")
print(submission_face["label"].value_counts().sort_index())

print("\nLinkuri de descărcare:")
display(FileLink(str(submission_face_path)))
display(FileLink(str(submission_face_alt_path)))
display(FileLink(str(zip_path)))

In [ ]:
import copy

FACE_FINAL_EPOCHS = 12          
FACE_FINAL_LR_BACKBONE = 3e-5
FACE_FINAL_LR_HEAD = 3e-4

print("Combin train + local_test pentru antrenarea finală...")

df_face_all = pd.concat(
    [df_face_train, df_face_local],
    ignore_index=True
)

face_all_ds = FaceTrainDataset(
    df_face_all,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=face_train_transform,
)

face_final_sampler, face_final_class_weights = make_sampler_and_weights(face_all_ds)

face_all_loader = DataLoader(
    face_all_ds,
    batch_size=FACE_BATCH_SIZE,
    sampler=face_final_sampler,
    num_workers=FACE_NUM_WORKERS,
    pin_memory=FACE_PIN_MEMORY,
    drop_last=True,
)


face_final_model = copy.deepcopy(face_model).to(DEVICE)


for param in face_final_model.parameters():
    param.requires_grad = True

backbone_params = []
head_params = []

for name, param in face_final_model.named_parameters():
    if not param.requires_grad:
        continue

    if name.startswith("classifier") or name.startswith("fc"):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": FACE_FINAL_LR_BACKBONE},
        {"params": head_params, "lr": FACE_FINAL_LR_HEAD},
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=FACE_FINAL_EPOCHS,
    eta_min=1e-6,
)

criterion_final = nn.CrossEntropyLoss(
    weight=face_final_class_weights.to(DEVICE),
    label_smoothing=0.03,
)

best_train_f1 = -1
best_final_state = None

print("Încep antrenarea finală pentru camera...")

for epoch in range(1, FACE_FINAL_EPOCHS + 1):
    train_metrics = train_one_epoch_face(
        face_final_model,
        face_all_loader,
        criterion_final,
        optimizer,
        freeze_batchnorm=False,
    )

    scheduler.step()

    print(
        f"FINAL CAMERA epoca {epoch}/{FACE_FINAL_EPOCHS} | "
        f"train F1={train_metrics['f1_macro']:.4f} | "
        f"loss={train_metrics['loss']:.4f}"
    )

    if train_metrics["f1_macro"] > best_train_f1:
        best_train_f1 = train_metrics["f1_macro"]
        best_final_state = {
            k: v.detach().cpu().clone()
            for k, v in face_final_model.state_dict().items()
        }

if best_final_state is not None:
    face_final_model.load_state_dict(best_final_state)

torch.save(face_final_model.state_dict(), WORK_DIR / "best_face_model_final_all_data.pt")

print("Gata. Model final salvat aici:")
print(WORK_DIR / "best_face_model_final_all_data.pt")

In [ ]:
face_remote_ds = FaceRemoteDataset(
    df_face_remote,
    FACE_REMOTE_ID_COL,
    transform=face_test_transform,
)

face_remote_loader = DataLoader(
    face_remote_ds,
    batch_size=FACE_BATCH_SIZE,
    shuffle=False,
    num_workers=FACE_NUM_WORKERS,
    pin_memory=FACE_PIN_MEMORY,
)


@torch.no_grad()
def predict_remote_face_strong(model, loader):
    model.eval()

    all_ids = []
    all_preds = []

    for images, sample_ids in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits_1 = model(images)
            logits_2 = model(torch.flip(images, dims=[3]))

            # TTA: media între imagine normală și flip orizontal.
            logits = (logits_1 + logits_2) / 2.0

        preds_idx = logits.argmax(dim=1).detach().cpu().numpy().tolist()

        all_ids.extend(list(sample_ids))
        all_preds.extend(preds_idx)

    return all_ids, all_preds


ids_face, preds_idx_face = predict_remote_face_strong(
    face_final_model,
    face_remote_loader,
)

preds_original_labels = [
    int(FACE_IDX_TO_LABEL[int(p)])
    for p in preds_idx_face
]

submission_face_final = pd.DataFrame({
    "id": ids_face,
    "label": preds_original_labels,
})

submission_face_final_path = WORK_DIR / "submission_face_final.csv"
submission_face_final.to_csv(submission_face_final_path, index=False)

print("Încarcă pe competiție acest fișier:")
print(submission_face_final_path)

display(submission_face_final.head())

print("Distribuție predicții:")
print(submission_face_final["label"].value_counts().sort_index())

display(FileLink(str(submission_face_final_path)))

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

WORK_DIR = Path("/kaggle/working")

csv_files = list(WORK_DIR.glob("*.csv"))

print("Fișiere CSV găsite în /kaggle/working:")
for f in csv_files:
    print("-", f.name)

print("\nLinkuri de descărcare:")
for f in csv_files:
    display(FileLink(str(f)))

In [ ]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, f1_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working")
USE_AMP = torch.cuda.is_available()

print("Device:", DEVICE)


df_face_all = pd.concat([df_face_train, df_face_local], ignore_index=True)


face_final_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.86, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.14,
        contrast=0.14,
        saturation=0.10,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.10),
])

face_final_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


face_final_train_ds = FaceTrainDataset(
    df_face_all,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=face_final_train_transform,
)

labels = np.array([label for _, label in face_final_train_ds.samples])
counts = np.bincount(labels, minlength=len(FACE_ORIGINAL_LABELS)).astype(np.float32)

print("Distribuție clase:")
for i, c in enumerate(counts.astype(int)):
    print(f"label {FACE_IDX_TO_LABEL[i]}: {c}")


sample_weights = (1.0 / np.sqrt(np.maximum(counts, 1.0)))[labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

class_weights = counts.sum() / (len(counts) * np.maximum(counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, 0.65, 2.2)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class weights:", class_weights.detach().cpu().numpy().round(3))

face_final_loader = DataLoader(
    face_final_train_ds,
    batch_size=32,         
    sampler=sampler,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)


def make_model(arch_name, num_classes):
    if arch_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(in_features, num_classes),
        )
        return model

    if arch_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(in_features, num_classes),
        )
        return model

    raise ValueError("Model necunoscut")


def is_head(name):
    return name.startswith("fc") or name.startswith("classifier")


def set_trainable(model, backbone_trainable):
    for name, param in model.named_parameters():
        if is_head(name):
            param.requires_grad = True
        else:
            param.requires_grad = backbone_trainable


def train_epoch(model, loader, criterion, optimizer):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "accuracy": accuracy_score(all_labels, all_preds),
    }


def train_final_model(arch_name, full_epochs=18):
    print("\n" + "=" * 60)
    print("ANTRENEZ:", arch_name)
    print("=" * 60)

    model = make_model(arch_name, len(FACE_ORIGINAL_LABELS)).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.03,
    )

    # Faza 1: doar head-ul
    set_trainable(model, backbone_trainable=False)

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1.2e-3,
        weight_decay=1e-4,
    )

    print("Faza 1: head only")

    for epoch in range(1, 3):
        metrics = train_epoch(model, face_final_loader, criterion, optimizer)
        print(
            f"{arch_name} HEAD epoca {epoch}/2 | "
            f"F1 train={metrics['f1_macro']:.4f} | loss={metrics['loss']:.4f}"
        )

  
    set_trainable(model, backbone_trainable=True)

    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if is_head(name):
            head_params.append(param)
        else:
            backbone_params.append(param)

    optimizer = optim.AdamW(
        [
            {"params": backbone_params, "lr": 4e-5},
            {"params": head_params, "lr": 4e-4},
        ],
        weight_decay=1e-4,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=full_epochs,
        eta_min=1e-6,
    )

    print("Faza 2: fine-tuning complet")

    best_state = None
    best_f1 = -1

    for epoch in range(1, full_epochs + 1):
        metrics = train_epoch(model, face_final_loader, criterion, optimizer)
        scheduler.step()

        print(
            f"{arch_name} FULL epoca {epoch}/{full_epochs} | "
            f"F1 train={metrics['f1_macro']:.4f} | loss={metrics['loss']:.4f}"
        )

        if metrics["f1_macro"] > best_f1:
            best_f1 = metrics["f1_macro"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    if best_state is not None:
        model.load_state_dict(best_state)

    path = WORK_DIR / f"{arch_name}_camera_final.pt"
    torch.save(model.state_dict(), path)

    print("Model salvat:", path)

    return model



camera_resnet18_final = train_final_model("resnet18", full_epochs=18)
camera_mobilenetv2_final = train_final_model("mobilenet_v2", full_epochs=18)

print("Gata antrenarea.")

In [ ]:
import zipfile
from IPython.display import display, FileLink

face_remote_final_ds = FaceRemoteDataset(
    df_face_remote,
    FACE_REMOTE_ID_COL,
    transform=face_final_test_transform,
)

face_remote_final_loader = DataLoader(
    face_remote_final_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    all_ids = []
    all_probs = []

    for images, sample_ids in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits1 = model(images)
            logits2 = model(torch.flip(images, dims=[3]))
            logits = (logits1 + logits2) / 2.0
            probs = torch.softmax(logits, dim=1)

        all_ids.extend(list(sample_ids))
        all_probs.append(probs.detach().cpu())

    probs = torch.cat(all_probs, dim=0).numpy()

    return all_ids, probs


def save_submission(ids, probs, filename):
    pred_idx = probs.argmax(axis=1)

    pred_labels = [
        int(FACE_IDX_TO_LABEL[int(p)])
        for p in pred_idx
    ]

    df_sub = pd.DataFrame({
        "id": ids,
        "label": pred_labels,
    })

    path = WORK_DIR / filename
    df_sub.to_csv(path, index=False)

    print("\nSalvat:", path)
    display(df_sub.head())
    print(df_sub["label"].value_counts().sort_index())

    return path


print("Prezic ResNet18...")
ids_resnet, probs_resnet = predict_model(
    camera_resnet18_final,
    face_remote_final_loader,
)

print("Prezic MobileNetV2...")
ids_mobile, probs_mobile = predict_model(
    camera_mobilenetv2_final,
    face_remote_final_loader,
)

if ids_resnet != ids_mobile:
    raise ValueError("ID-urile nu sunt în aceeași ordine.")


probs_ensemble = (probs_resnet + probs_mobile) / 2.0

path_ensemble = save_submission(
    ids_resnet,
    probs_ensemble,
    "submission_face_ensemble_final.csv",
)

path_resnet = save_submission(
    ids_resnet,
    probs_resnet,
    "submission_face_resnet18_final.csv",
)

path_mobile = save_submission(
    ids_mobile,
    probs_mobile,
    "submission_face_mobilenetv2_final.csv",
)

zip_path = WORK_DIR / "camera_final_submissions.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    z.write(path_ensemble, arcname=path_ensemble.name)
    z.write(path_resnet, arcname=path_resnet.name)
    z.write(path_mobile, arcname=path_mobile.name)

print("\n====================================")
print("ÎNCARCĂ PRIMA DATĂ:")
print(path_ensemble)
print("====================================")

print("\nDacă nu trece, încearcă apoi:")
print(path_resnet)
print(path_mobile)

display(FileLink(str(path_ensemble)))
display(FileLink(str(path_resnet)))
display(FileLink(str(path_mobile)))
display(FileLink(str(zip_path)))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.metrics import f1_score, accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working")
USE_AMP = torch.cuda.is_available()

print("Device:", DEVICE)


needed = [
    "df_face_train",
    "df_face_local",
    "df_face_remote",
    "FaceTrainDataset",
    "FaceRemoteDataset",
    "FACE_PATH_COL",
    "FACE_LABEL_COL",
    "FACE_REMOTE_ID_COL",
    "FACE_ORIGINAL_LABELS",
    "FACE_IDX_TO_LABEL",
]

missing = [x for x in needed if x not in globals()]
if missing:
    raise NameError("Lipsesc: " + ", ".join(missing) + ". Rulează întâi setup-ul pentru camera.")

df_face_turbo = pd.concat(
    [df_face_train, df_face_local],
    ignore_index=True
)

TURBO_IMAGE_SIZE = 260
TURBO_BATCH_SIZE = 16
TURBO_HEAD_EPOCHS = 2
TURBO_FULL_EPOCHS = 24   

turbo_train_transform = transforms.Compose([
    transforms.Resize((288, 288)),
    transforms.RandomResizedCrop(TURBO_IMAGE_SIZE, scale=(0.88, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.12,
        contrast=0.12,
        saturation=0.08,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
    transforms.RandomErasing(p=0.08),
])

turbo_test_transform = transforms.Compose([
    transforms.Resize((TURBO_IMAGE_SIZE, TURBO_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
])

turbo_train_ds = FaceTrainDataset(
    df_face_turbo,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=turbo_train_transform,
)

labels = np.array([label for _, label in turbo_train_ds.samples])
counts = np.bincount(labels, minlength=len(FACE_ORIGINAL_LABELS)).astype(np.float32)

print("Distribuție clase:")
for idx, cnt in enumerate(counts.astype(int)):
    print(f"label {FACE_IDX_TO_LABEL[idx]}: {cnt}")


sample_weights = (1.0 / np.sqrt(np.maximum(counts, 1.0)))[labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)


class_weights = counts.sum() / (len(counts) * np.maximum(counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, 0.75, 1.90)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class weights:", class_weights.detach().cpu().numpy().round(3))

turbo_train_loader = DataLoader(
    turbo_train_ds,
    batch_size=TURBO_BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)


def build_turbo_model(num_classes):
    try:
        model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(in_features, num_classes),
        )
        print("Model folosit: EfficientNet-B2 pretrained")
        return model
    except Exception as e:
        print("EfficientNet-B2 nu a mers, încerc EfficientNet-B0:", repr(e))

    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.35),
        nn.Linear(in_features, num_classes),
    )
    print("Model folosit: EfficientNet-B0 pretrained")
    return model


def set_backbone_trainable(model, trainable):
    for name, param in model.named_parameters():
        if name.startswith("classifier"):
            param.requires_grad = True
        else:
            param.requires_grad = trainable


def train_one_epoch_turbo(model, loader, criterion, optimizer):
    model.train()

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "accuracy": accuracy_score(all_labels, all_preds),
    }


turbo_model = build_turbo_model(len(FACE_ORIGINAL_LABELS)).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.025,
)


print("\nFaza 1: antrenez doar classifier-ul")

set_backbone_trainable(turbo_model, trainable=False)

optimizer = optim.AdamW(
    [p for p in turbo_model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=1e-4,
)

for epoch in range(1, TURBO_HEAD_EPOCHS + 1):
    m = train_one_epoch_turbo(turbo_model, turbo_train_loader, criterion, optimizer)
    print(f"HEAD epoca {epoch}/{TURBO_HEAD_EPOCHS} | F1 train={m['f1_macro']:.4f} | loss={m['loss']:.4f}")


print("\nFaza 2: fine-tuning complet")

set_backbone_trainable(turbo_model, trainable=True)

backbone_params = []
head_params = []

for name, param in turbo_model.named_parameters():
    if name.startswith("classifier"):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": 2.5e-5},
        {"params": head_params, "lr": 2.5e-4},
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TURBO_FULL_EPOCHS,
    eta_min=1e-6,
)

for epoch in range(1, TURBO_FULL_EPOCHS + 1):
    m = train_one_epoch_turbo(turbo_model, turbo_train_loader, criterion, optimizer)
    scheduler.step()

    print(f"TURBO epoca {epoch}/{TURBO_FULL_EPOCHS} | F1 train={m['f1_macro']:.4f} | loss={m['loss']:.4f}")

torch.save(turbo_model.state_dict(), WORK_DIR / "camera_turbo_model.pt")

print("\nGata. Model salvat:")
print(WORK_DIR / "camera_turbo_model.pt")

In [ ]:
import zipfile
from IPython.display import display, FileLink

turbo_remote_ds = FaceRemoteDataset(
    df_face_remote,
    FACE_REMOTE_ID_COL,
    transform=turbo_test_transform,
)

turbo_remote_loader = DataLoader(
    turbo_remote_ds,
    batch_size=TURBO_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


@torch.no_grad()
def predict_probs_turbo(model, loader):
    model.eval()

    all_ids = []
    all_probs = []

    for images, sample_ids in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits_1 = model(images)
            logits_2 = model(torch.flip(images, dims=[3]))
            logits = (logits_1 + logits_2) / 2.0
            probs = torch.softmax(logits, dim=1)

        all_ids.extend(list(sample_ids))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()
    return all_ids, all_probs


def save_face_submission(ids, probs, filename):
    pred_idx = probs.argmax(axis=1)

    pred_labels = [
        int(FACE_IDX_TO_LABEL[int(p)])
        for p in pred_idx
    ]

    df_sub = pd.DataFrame({
        "id": ids,
        "label": pred_labels,
    })

    path = WORK_DIR / filename
    df_sub.to_csv(path, index=False)

    print("\nSalvat:", path)
    display(df_sub.head())

    print("\nDistribuție predicții:")
    print(df_sub["label"].value_counts().sort_index())

    return path


ids_turbo, probs_turbo = predict_probs_turbo(
    turbo_model,
    turbo_remote_loader,
)

path_turbo = save_face_submission(
    ids_turbo,
    probs_turbo,
    "submission_face_turbo.csv",
)

probs_list = [probs_turbo]
names = ["turbo"]

for model_name in ["face_final_model", "camera_resnet18_final", "camera_mobilenetv2_final", "face_model"]:
    if model_name in globals():
        try:
            print(f"Adaug în ensemble: {model_name}")
            _, p = predict_probs_turbo(globals()[model_name].to(DEVICE), turbo_remote_loader)
            probs_list.append(p)
            names.append(model_name)
        except Exception as e:
            print(f"Nu pot folosi {model_name}:", repr(e))

if len(probs_list) > 1:
    probs_ensemble = np.mean(probs_list, axis=0)

    path_ensemble = save_face_submission(
        ids_turbo,
        probs_ensemble,
        "submission_face_turbo_ensemble.csv",
    )
else:
    path_ensemble = None

zip_path = WORK_DIR / "camera_turbo_submissions.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    z.write(path_turbo, arcname=path_turbo.name)
    if path_ensemble is not None:
        z.write(path_ensemble, arcname=path_ensemble.name)

print("\n====================================")
print("ÎNCARCĂ PRIMA DATĂ:")
if path_ensemble is not None:
    print(path_ensemble)
else:
    print(path_turbo)
print("====================================")

print("\nDacă scade scorul, păstrezi best-ul vechi 0.75224.")
print("Modele în ensemble:", names)

display(FileLink(str(path_turbo)))
if path_ensemble is not None:
    display(FileLink(str(path_ensemble)))
display(FileLink(str(zip_path)))

In [ ]:
from pathlib import Path
import pandas as pd
import json
from IPython.display import display

WORK_DIR = Path("/kaggle/working")


SKIN_KAGGLE_SCORE = 0.70157      
CAMERA_KAGGLE_SCORE = 0.75224    



thresholds = pd.DataFrame([
    {
        "Dataset": "Vai de pielea mea",
        "Model / cerință": "MLP",
        "Prag F1-macro": 0.20,
        "Scor Kaggle": SKIN_KAGGLE_SCORE,
    },
    {
        "Dataset": "Vai de pielea mea",
        "Model / cerință": "CNN",
        "Prag F1-macro": 0.40,
        "Scor Kaggle": SKIN_KAGGLE_SCORE,
    },
    {
        "Dataset": "Vai de pielea mea",
        "Model / cerință": "Finetuned ResNet18",
        "Prag F1-macro": 0.67,
        "Scor Kaggle": SKIN_KAGGLE_SCORE,
    },
    {
        "Dataset": "Vai de pielea mea",
        "Model / cerință": "Finetuned MobileNet-v2",
        "Prag F1-macro": 0.65,
        "Scor Kaggle": SKIN_KAGGLE_SCORE,
    },
    {
        "Dataset": "You're on candid camera",
        "Model / cerință": "MLP",
        "Prag F1-macro": 0.19,
        "Scor Kaggle": CAMERA_KAGGLE_SCORE,
    },
    {
        "Dataset": "You're on candid camera",
        "Model / cerință": "CNN",
        "Prag F1-macro": 0.35,
        "Scor Kaggle": CAMERA_KAGGLE_SCORE,
    },
    {
        "Dataset": "You're on candid camera",
        "Model / cerință": "ResNet18",
        "Prag F1-macro": 0.77,
        "Scor Kaggle": CAMERA_KAGGLE_SCORE,
    },
    {
        "Dataset": "You're on candid camera",
        "Model / cerință": "MobileNetV2",
        "Prag F1-macro": 0.76,
        "Scor Kaggle": CAMERA_KAGGLE_SCORE,
    },
])

thresholds["Diferență față de prag"] = thresholds["Scor Kaggle"] - thresholds["Prag F1-macro"]
thresholds["Status"] = thresholds["Diferență față de prag"].apply(
    lambda x: " TRECE" if x >= 0 else " NU TRECE"
)

print("=" * 80)
print("VERIFICARE PRAGURI KAGGLE")
print("=" * 80)

display(thresholds)



summary = thresholds.groupby("Dataset").agg(
    Scor_Kaggle=("Scor Kaggle", "first"),
    Prag_maxim=("Prag F1-macro", "max"),
    Trecute=("Status", lambda s: (s == " TRECE").sum()),
    Total=("Status", "count"),
).reset_index()

summary["Status general"] = summary.apply(
    lambda row: " TOATE PRAGURILE TRECUTE" if row["Trecute"] == row["Total"] else " NU TREC TOATE",
    axis=1
)

print("\n" + "=" * 80)
print("REZUMAT PE DATASET")
print("=" * 80)

display(summary)



expected_files = [
    {
        "Dataset": "Vai de pielea mea",
        "Fișier": "submission_mole.csv",
        "Rol": "submission principal skin",
    },
    {
        "Dataset": "Vai de pielea mea",
        "Fișier": "submission_mole_labels.csv",
        "Rol": "variantă cu label text skin",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face.csv",
        "Rol": "submission camera",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face_final.csv",
        "Rol": "submission camera boost final",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face_prediction.csv",
        "Rol": "variantă Id / Prediction",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face_turbo.csv",
        "Rol": "variantă turbo",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face_turbo_ensemble.csv",
        "Rol": "variantă turbo ensemble",
    },
    {
        "Dataset": "You're on candid camera",
        "Fișier": "submission_face_ensemble_final.csv",
        "Rol": "variantă ensemble final",
    },
]

file_rows = []

for item in expected_files:
    path = WORK_DIR / item["Fișier"]
    exists = path.exists()
    
    file_rows.append({
        "Dataset": item["Dataset"],
        "Fișier": item["Fișier"],
        "Rol": item["Rol"],
        "Există în /kaggle/working": " DA" if exists else " NU",
        "Path": str(path) if exists else "-",
    })

files_df = pd.DataFrame(file_rows)

print("\n" + "=" * 80)
print("VERIFICARE FIȘIERE DE SUBMISSION")
print("=" * 80)

display(files_df)


all_csvs = sorted(WORK_DIR.rglob("*.csv"))

csv_rows = []

for path in all_csvs:
    try:
        df_tmp = pd.read_csv(path, nrows=5)
        shape_info = f"{len(pd.read_csv(path))} rânduri, {len(df_tmp.columns)} coloane"
        columns_info = ", ".join(list(df_tmp.columns))
    except Exception:
        shape_info = "nu pot citi"
        columns_info = "-"

    csv_rows.append({
        "Fișier CSV găsit": path.name,
        "Folder": str(path.parent),
        "Dimensiune": shape_info,
        "Coloane": columns_info,
    })

all_csvs_df = pd.DataFrame(csv_rows)

print("\n" + "=" * 80)
print("TOATE CSV-URILE GĂSITE")
print("=" * 80)

display(all_csvs_df)



metric_rows = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    # Caută în dicționare de tip result
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_lower = str(key).lower()
            if "f1" in key_lower:
                try:
                    metric_rows.append({
                        "Sursă": name,
                        "Metrică": key,
                        "Valoare": float(value),
                    })
                except Exception:
                    pass

    # Caută în DataFrame-uri de tip history
    if isinstance(obj, pd.DataFrame):
        for col in obj.columns:
            col_lower = str(col).lower()
            if "f1" in col_lower:
                try:
                    metric_rows.append({
                        "Sursă": name,
                        "Metrică": f"{col} max",
                        "Valoare": float(obj[col].max()),
                    })
                except Exception:
                    pass

metric_df = pd.DataFrame(metric_rows)

print("\n" + "=" * 80)
print("METRICI F1 GĂSITE ÎN NOTEBOOK")
print("=" * 80)

if len(metric_df) == 0:
    print("Nu am găsit metrici F1 în variabilele curente.")
else:
    metric_df = metric_df.sort_values("Valoare", ascending=False)
    display(metric_df)



print("\n" + "=" * 80)
print("CONCLUZIE")
print("=" * 80)

for _, row in summary.iterrows():
    print(f"{row['Dataset']}: {row['Status general']} — scor Kaggle = {row['Scor_Kaggle']:.5f}, prag maxim = {row['Prag_maxim']:.2f}")

print("\nFișiere recomandate pentru încărcare:")

print("- Skin / Vai de pielea mea:")
if (WORK_DIR / "submission_mole.csv").exists():
    print("   /kaggle/working/submission_mole.csv")
else:
    print("   lipsește submission_mole.csv")

print("- Camera / You're on candid camera:")
camera_candidates = [
    "submission_face_turbo_ensemble.csv",
    "submission_face_turbo.csv",
    "submission_face_ensemble_final.csv",
    "submission_face_final.csv",
    "submission_face.csv",
]

found_camera = False
for fname in camera_candidates:
    if (WORK_DIR / fname).exists():
        print(f"   /kaggle/working/{fname}")
        found_camera = True

if not found_camera:
    print("   nu am găsit niciun submission pentru camera")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, FileLink

import torch
from torch.utils.data import DataLoader

WORK_DIR = Path("/kaggle/working")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

REMOTE_QUOTAS_ORIGINAL_LABELS = {
    1: 329,   # Surprise
    2: 74,    # Fear
    3: 160,   # Disgust
    4: 1185,  # Happiness
    5: 478,   # Sadness
    6: 162,   # Anger
    7: 680,   # Neutral
}

print("Total quota:", sum(REMOTE_QUOTAS_ORIGINAL_LABELS.values()))



if "turbo_test_transform" in globals():
    quota_test_transform = turbo_test_transform
    quota_batch_size = globals().get("TURBO_BATCH_SIZE", 16)
    print("Folosesc turbo_test_transform")
elif "face_final_test_transform" in globals():
    quota_test_transform = face_final_test_transform
    quota_batch_size = 32
    print("Folosesc face_final_test_transform")
elif "face_strong_test_transform" in globals():
    quota_test_transform = face_strong_test_transform
    quota_batch_size = 32
    print("Folosesc face_strong_test_transform")
elif "face_test_transform" in globals():
    quota_test_transform = face_test_transform
    quota_batch_size = globals().get("FACE_BATCH_SIZE", 32)
    print("Folosesc face_test_transform")
else:
    raise NameError("Nu găsesc nicio transformare de test pentru camera.")



quota_remote_ds = FaceRemoteDataset(
    df_face_remote,
    FACE_REMOTE_ID_COL,
    transform=quota_test_transform,
)

quota_remote_loader = DataLoader(
    quota_remote_ds,
    batch_size=quota_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



@torch.no_grad()
def predict_probs_for_quota(model, loader):
    model.eval()
    model = model.to(DEVICE)

    all_ids = []
    all_probs = []

    for images, sample_ids in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits1 = model(images)
            logits2 = model(torch.flip(images, dims=[3]))
            logits = (logits1 + logits2) / 2.0
            probs = torch.softmax(logits, dim=1)

        all_ids.extend(list(sample_ids))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()
    return all_ids, all_probs


models_for_quota = []

# punem modelele care au cele mai mari șanse
candidate_model_names = [
    "turbo_model",
    "face_final_model",
    "camera_resnet18_final",
    "camera_mobilenetv2_final",
    "face_model",
]

for name in candidate_model_names:
    if name in globals():
        models_for_quota.append((name, globals()[name]))

if len(models_for_quota) == 0:
    raise NameError(
        "Nu găsesc niciun model în memorie. Trebuie să existe măcar face_model, "
        "face_final_model, turbo_model, camera_resnet18_final sau camera_mobilenetv2_final."
    )

print("Modele folosite:")
for name, _ in models_for_quota:
    print("-", name)


all_probs = []
all_names = []
ids_ref = None

for name, model in models_for_quota:
    print(f"\nPrezic cu {name}...")
    ids, probs = predict_probs_for_quota(model, quota_remote_loader)

    if ids_ref is None:
        ids_ref = ids
    else:
        if ids != ids_ref:
            raise ValueError(f"ID-urile nu sunt în aceeași ordine pentru modelul {name}")

    all_probs.append(probs)
    all_names.append(name)



weights = []

for name in all_names:
    if name == "turbo_model":
        weights.append(2.0)
    elif name == "face_final_model":
        weights.append(2.0)
    else:
        weights.append(1.0)

weights = np.array(weights, dtype=np.float32)
weights = weights / weights.sum()

print("\nPonderi ensemble:")
for name, w in zip(all_names, weights):
    print(f"{name}: {w:.3f}")

probs_ensemble = np.zeros_like(all_probs[0])

for p, w in zip(all_probs, weights):
    probs_ensemble += w * p


quotas_idx = {}

for original_label, count in REMOTE_QUOTAS_ORIGINAL_LABELS.items():
    idx = None

    if "FACE_LABEL_TO_IDX" in globals():
        idx = FACE_LABEL_TO_IDX[int(original_label)]
    else:
        # fallback: caută în FACE_IDX_TO_LABEL
        for k, v in FACE_IDX_TO_LABEL.items():
            if int(v) == int(original_label):
                idx = int(k)
                break

    if idx is None:
        raise ValueError(f"Nu pot mapa labelul original {original_label} la index model.")

    quotas_idx[idx] = count

print("\nQuota pe index model:")
print(quotas_idx)

if sum(quotas_idx.values()) != len(ids_ref):
    raise ValueError(
        f"Quota totală este {sum(quotas_idx.values())}, dar remote_test are {len(ids_ref)} imagini."
    )


def assign_with_exact_quotas(probs, quotas_idx):
    """
    Atribuire exactă pe clase folosind linear_sum_assignment.
    Dacă scipy nu merge, folosește fallback greedy.
    """

    eps = 1e-12
    log_probs = np.log(np.clip(probs, eps, 1.0))

    n, c = probs.shape

    class_slots = []
    for cls_idx, count in sorted(quotas_idx.items()):
        class_slots.extend([cls_idx] * count)

    class_slots = np.array(class_slots, dtype=np.int64)

    if len(class_slots) != n:
        raise ValueError("Numărul de sloturi nu coincide cu numărul de imagini.")

    try:
        from scipy.optimize import linear_sum_assignment

        # cost: vrem să maximizăm log_prob, deci minimizăm -log_prob
        cost = -log_probs[:, class_slots].astype(np.float32)
        row_ind, col_ind = linear_sum_assignment(cost)

        assigned = np.empty(n, dtype=np.int64)
        assigned[row_ind] = class_slots[col_ind]

        print("Atribuire exactă făcută cu scipy linear_sum_assignment.")
        return assigned

    except Exception as e:
        print("Scipy nu a mers, folosesc fallback greedy:", repr(e))

        assigned = probs.argmax(axis=1).astype(np.int64)

        def counts_now():
            return {cls: int((assigned == cls).sum()) for cls in quotas_idx}

        for _ in range(10000):
            current = counts_now()

            over = [cls for cls in quotas_idx if current[cls] > quotas_idx[cls]]
            under = [cls for cls in quotas_idx if current[cls] < quotas_idx[cls]]

            if not over and not under:
                break

            best_move = None
            best_penalty = float("inf")

            for src in over:
                src_indices = np.where(assigned == src)[0]

                for dst in under:
                    penalties = log_probs[src_indices, src] - log_probs[src_indices, dst]
                    j = penalties.argmin()
                    penalty = penalties[j]

                    if penalty < best_penalty:
                        best_penalty = penalty
                        best_move = (src_indices[j], dst)

            if best_move is None:
                break

            sample_idx, dst = best_move
            assigned[sample_idx] = dst

        return assigned


assigned_idx = assign_with_exact_quotas(probs_ensemble, quotas_idx)



assigned_original_labels = [
    int(FACE_IDX_TO_LABEL[int(idx)])
    for idx in assigned_idx
]

submission_quota = pd.DataFrame({
    "id": ids_ref,
    "label": assigned_original_labels,
})

quota_path = WORK_DIR / "submission_face_quota.csv"
submission_quota.to_csv(quota_path, index=False)

print("\nSalvat:")
print(quota_path)

print("\nPreview:")
display(submission_quota.head())

print("\nDistribuție predicții finale:")
print(submission_quota["label"].value_counts().sort_index())

print("\nTrebuie să fie:")
print(pd.Series(REMOTE_QUOTAS_ORIGINAL_LABELS).sort_index())

display(FileLink(str(quota_path)))

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink

WORK_DIR = Path("/kaggle/working")

quota_files = list(WORK_DIR.rglob("*quota*.csv"))

print("Fișiere quota găsite:")
for f in quota_files:
    print(f)

if len(quota_files) == 0:
    print("\n Nu există submission_face_quota.csv.")
    print("Asta înseamnă că celula CAMERA QUOTA BOOST nu a rulat până la capăt.")
else:
    print("\n Am găsit:")
    for f in quota_files:
        display(FileLink(str(f)))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    classification_report,
)

from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working")
CONF_DIR = WORK_DIR / "confusion_matrices_finale"
CONF_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = torch.cuda.is_available()

print("Device:", DEVICE)
print("Matricile se salvează în:", CONF_DIR)


def evaluate_and_show_confusion(
    model,
    test_loader,
    class_labels,
    experiment_name="Experiment",
    use_tta=False,
    save_png=True,
):
    """
    Evaluează un model pe un loader CU ETICHETE și afișează:
    - Accuracy
    - F1-macro
    - F1-micro
    - matrice de confuzie cu valori absolute
    - matrice de confuzie normalizată pe clasa reală

    IMPORTANT:
    Merge doar pe test/local_test cu etichete.
    Nu merge pe remote_test Kaggle, fiindcă acolo nu există etichete reale.
    """

    model = model.to(DEVICE)
    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():
        for batch in test_loader:
            images, labels = batch[0], batch[1]

            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images)

                # TTA simplu: predicție normală + flip orizontal
                if use_tta:
                    logits_flip = model(torch.flip(images, dims=[3]))
                    logits = (logits + logits_flip) / 2.0

            preds = logits.argmax(dim=1)

            all_true.extend(labels.detach().cpu().numpy())
            all_pred.extend(preds.detach().cpu().numpy())

    y_true = np.array(all_true)
    y_pred = np.array(all_pred)

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    result = {
        "experiment": experiment_name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
    }

    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)
    print(f"Accuracy : {acc:.5f}")
    print(f"F1-macro : {f1_macro:.5f}")
    print(f"F1-micro : {f1_micro:.5f}")

    labels_idx = list(range(len(class_labels)))
    cm = confusion_matrix(y_true, y_pred, labels=labels_idx)

    # --------------------------------------------------------
    # Matrice valori absolute
    # --------------------------------------------------------
    fig_size = max(7, len(class_labels) * 1.15)

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(cm)

    ax.set_title(f"{experiment_name}\nMatrice de confuzie — valori absolute")
    ax.set_xlabel("Clasa prezisă")
    ax.set_ylabel("Clasa reală")

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticklabels(class_labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=9,
            )

    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    safe_name = (
        experiment_name
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("—", "_")
        .replace("-", "_")
    )

    counts_path = CONF_DIR / f"{safe_name}_confusion_counts.png"

    if save_png:
        plt.savefig(counts_path, dpi=160, bbox_inches="tight")

    plt.show()

    # --------------------------------------------------------
    # Matrice normalizată
    # --------------------------------------------------------
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(cm_norm)

    ax.set_title(f"{experiment_name}\nMatrice de confuzie — normalizată")
    ax.set_xlabel("Clasa prezisă")
    ax.set_ylabel("Clasa reală")

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticklabels(class_labels)

    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm_norm[i, j] * 100:.1f}%",
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    norm_path = CONF_DIR / f"{safe_name}_confusion_normalized.png"

    if save_png:
        plt.savefig(norm_path, dpi=160, bbox_inches="tight")

    plt.show()

    # --------------------------------------------------------
    # Classification report tabel
    # --------------------------------------------------------
    report = classification_report(
        y_true,
        y_pred,
        labels=labels_idx,
        target_names=[str(x) for x in class_labels],
        zero_division=0,
        output_dict=True,
    )

    report_df = pd.DataFrame(report).transpose()

    print("\nClassification report:")
    display(report_df)

    report_path = CONF_DIR / f"{safe_name}_classification_report.csv"
    report_df.to_csv(report_path)

    print("\nFișiere salvate:")
    print(counts_path)
    print(norm_path)
    print(report_path)

    result["confusion_counts_path"] = str(counts_path)
    result["confusion_normalized_path"] = str(norm_path)
    result["classification_report_path"] = str(report_path)

    return result

In [ ]:
cnn_skin_confusion_result = evaluate_and_show_confusion(
    model=cnn_skin_model,
    test_loader=skin_test_cnn_loader,
    class_labels=skin_class_labels,
    experiment_name="CNN skin cu BatchNorm",
    use_tta=False,
)

cnn_skin_confusion_result

In [ ]:
cnn_camera_confusion_result = evaluate_and_show_confusion(
    model=cnn_camera_model,
    test_loader=camera_test_cnn_loader,
    class_labels=camera_class_labels,
    experiment_name="CNN camera cu BatchNorm",
    use_tta=False,
)

cnn_camera_confusion_result

In [ ]:
strong_skin_confusion_result = evaluate_and_show_confusion(
    model=strong_skin_model,
    test_loader=skin_test_boost_loader,
    class_labels=skin_class_labels,
    experiment_name="Skin strong boosted model",
    use_tta=True,
)

strong_skin_confusion_result

In [ ]:
from torch.utils.data import DataLoader

if "face_local_loader" not in globals() and "face_test_transform" in globals():
    face_local_ds_for_eval = FaceTrainDataset(
        df_face_local,
        FACE_PATH_COL,
        FACE_LABEL_COL,
        transform=face_test_transform,
    )

    face_local_loader = DataLoader(
        face_local_ds_for_eval,
        batch_size=globals().get("FACE_BATCH_SIZE", 32),
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    print("Creat face_local_loader")


if "turbo_local_loader" not in globals() and "turbo_test_transform" in globals():
    turbo_local_ds_for_eval = FaceTrainDataset(
        df_face_local,
        FACE_PATH_COL,
        FACE_LABEL_COL,
        transform=turbo_test_transform,
    )

    turbo_local_loader = DataLoader(
        turbo_local_ds_for_eval,
        batch_size=globals().get("TURBO_BATCH_SIZE", 16),
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    print("Creat turbo_local_loader")

In [ ]:
from torch.utils.data import DataLoader

face_local_ds_for_eval = FaceTrainDataset(
    df_face_local,
    FACE_PATH_COL,
    FACE_LABEL_COL,
    transform=face_test_transform,
)

face_local_loader_safe = DataLoader(
    face_local_ds_for_eval,
    batch_size=16,       # mai sigur decât 32
    shuffle=False,
    num_workers=0,       # important
    pin_memory=False,    # important
)

print("Creat face_local_loader_safe:", len(face_local_ds_for_eval), "imagini")

In [ ]:
face_final_confusion_result = evaluate_and_show_confusion(
    model=face_final_model,
    test_loader=face_local_loader_safe,
    class_labels=get_camera_class_labels() if "get_camera_class_labels" in globals() else camera_class_labels,
    experiment_name="Camera final model",
    use_tta=False,   # mai rapid; pune True doar dacă ai timp
)

face_final_confusion_result

In [ ]:
# ============================================================
# Matrice de confuzie — Camera turbo model
# ============================================================

turbo_confusion_result = evaluate_and_show_confusion(
    model=turbo_model,
    test_loader=turbo_local_loader,
    class_labels=get_camera_class_labels() if "get_camera_class_labels" in globals() else camera_class_labels,
    experiment_name="Camera turbo model",
    use_tta=True,
)

turbo_confusion_result

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
from IPython.display import display, FileLink

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working")
CM_DIR = WORK_DIR / "matrici_confuzie_modele_bune"
CM_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = torch.cuda.is_available()

print("Device:", DEVICE)
print("Salvez matricile în:", CM_DIR)


def get_camera_labels_for_plot():
    if "camera_class_labels" in globals():
        return camera_class_labels

    if "FACE_IDX_TO_LABEL" in globals():
        labels = []
        for i in range(len(FACE_IDX_TO_LABEL)):
            original_label = int(FACE_IDX_TO_LABEL[i])
            if "FACE_CLASS_NAMES" in globals():
                labels.append(f"{original_label}-{FACE_CLASS_NAMES.get(original_label, str(original_label))}")
            else:
                labels.append(str(original_label))
        return labels

    return [str(i) for i in range(7)]


def get_skin_labels_for_plot():
    if "skin_class_labels" in globals():
        return skin_class_labels

    if "skin_idx_to_class" in globals():
        return [str(skin_idx_to_class[i]) for i in range(len(skin_idx_to_class))]

    if "skin_class_to_idx" in globals():
        inv = {v: k for k, v in skin_class_to_idx.items()}
        return [str(inv[i]) for i in range(len(inv))]

    return [str(i) for i in range(7)]


def safe_filename(text):
    return (
        str(text)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("—", "_")
        .replace("-", "_")
        .replace("__", "_")
    )


@torch.no_grad()
def predict_for_confusion(model, loader, use_tta=False):
    model = model.to(DEVICE)
    model.eval()

    y_true = []
    y_pred = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)

            if use_tta:
                logits_flip = model(torch.flip(images, dims=[3]))
                logits = (logits + logits_flip) / 2.0

        preds = logits.argmax(dim=1)

        y_true.extend(labels.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

    return np.array(y_true), np.array(y_pred)


def show_confusion_matrix_good_model(
    model,
    loader,
    class_labels,
    title,
    use_tta=False,
):
    y_true, y_pred = predict_for_confusion(
        model=model,
        loader=loader,
        use_tta=use_tta,
    )

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print(f"Accuracy : {acc:.5f}")
    print(f"F1-macro : {f1_macro:.5f}")
    print(f"F1-micro : {f1_micro:.5f}")

    labels_idx = list(range(len(class_labels)))
    cm = confusion_matrix(y_true, y_pred, labels=labels_idx)

    # Matrice valori absolute
    fig_size = max(7, len(class_labels) * 1.15)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    im = ax.imshow(cm)
    ax.set_title(f"{title}\nMatrice de confuzie — valori absolute")
    ax.set_xlabel("Clasa prezisă")
    ax.set_ylabel("Clasa reală")

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticklabels(class_labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    fname = safe_filename(title)
    counts_path = CM_DIR / f"{fname}_confusion_counts.png"
    plt.savefig(counts_path, dpi=160, bbox_inches="tight")
    plt.show()

    # Matrice normalizată
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    im = ax.imshow(cm_norm)
    ax.set_title(f"{title}\nMatrice de confuzie — normalizată pe clasa reală")
    ax.set_xlabel("Clasa prezisă")
    ax.set_ylabel("Clasa reală")

    ax.set_xticks(np.arange(len(class_labels)))
    ax.set_yticks(np.arange(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticklabels(class_labels)

    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(j, i, f"{cm_norm[i, j] * 100:.1f}%", ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    norm_path = CM_DIR / f"{fname}_confusion_normalized.png"
    plt.savefig(norm_path, dpi=160, bbox_inches="tight")
    plt.show()

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=labels_idx,
            target_names=[str(x) for x in class_labels],
            zero_division=0,
            output_dict=True,
        )
    ).transpose()

    report_path = CM_DIR / f"{fname}_classification_report.csv"
    report_df.to_csv(report_path)

    print("Classification report:")
    display(report_df)

    result = {
        "experiment": title,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "confusion_counts": str(counts_path),
        "confusion_normalized": str(norm_path),
        "classification_report": str(report_path),
    }

    print("Fișiere salvate:")
    print(counts_path)
    print(norm_path)
    print(report_path)

    return result


skin_labels = get_skin_labels_for_plot()
camera_labels = get_camera_labels_for_plot()

# Skin loader pentru modelul bun nou
skin_good_loader = None

if "skin_test_boost_ds" in globals():
    skin_good_loader = DataLoader(
        skin_test_boost_ds,
        batch_size=16,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )
elif "skin_test_boost_loader" in globals():
    skin_good_loader = skin_test_boost_loader
elif "skin_test_ft_loader" in globals():
    skin_good_loader = skin_test_ft_loader
elif "skin_test_cnn_loader" in globals():
    skin_good_loader = skin_test_cnn_loader


# Camera loader pentru face_final_model
camera_face_loader = None

if "FaceTrainDataset" in globals() and "df_face_local" in globals():
    if "face_test_transform" in globals():
        camera_face_ds = FaceTrainDataset(
            df_face_local,
            FACE_PATH_COL,
            FACE_LABEL_COL,
            transform=face_test_transform,
        )

        camera_face_loader = DataLoader(
            camera_face_ds,
            batch_size=16,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        )


# Camera loader pentru turbo_model
camera_turbo_loader = None

if "FaceTrainDataset" in globals() and "df_face_local" in globals():
    if "turbo_test_transform" in globals():
        camera_turbo_ds = FaceTrainDataset(
            df_face_local,
            FACE_PATH_COL,
            FACE_LABEL_COL,
            transform=turbo_test_transform,
        )

        camera_turbo_loader = DataLoader(
            camera_turbo_ds,
            batch_size=16,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        )


# Camera loader pentru ResNet18/MobileNetV2 final
camera_final_loader = None

if "FaceTrainDataset" in globals() and "df_face_local" in globals():
    if "face_final_test_transform" in globals():
        camera_final_ds = FaceTrainDataset(
            df_face_local,
            FACE_PATH_COL,
            FACE_LABEL_COL,
            transform=face_final_test_transform,
        )

        camera_final_loader = DataLoader(
            camera_final_ds,
            batch_size=16,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
        )


results_good_confusions = []

# 1. Skin — modelul nou bun
if "strong_skin_model" in globals() and skin_good_loader is not None:
    results_good_confusions.append(
        show_confusion_matrix_good_model(
            model=strong_skin_model,
            loader=skin_good_loader,
            class_labels=skin_labels,
            title="Skin — strong model final nou",
            use_tta=True,
        )
    )
else:
    print("Nu pot face matrice pentru strong_skin_model: lipsește modelul sau loaderul.")


# 2. Camera — model final obișnuit
if "face_final_model" in globals() and camera_face_loader is not None:
    results_good_confusions.append(
        show_confusion_matrix_good_model(
            model=face_final_model,
            loader=camera_face_loader,
            class_labels=camera_labels,
            title="Camera — face_final_model nou",
            use_tta=True,
        )
    )
else:
    print("Nu pot face matrice pentru face_final_model: lipsește modelul sau loaderul.")


# 3. Camera — turbo model
if "turbo_model" in globals() and camera_turbo_loader is not None:
    results_good_confusions.append(
        show_confusion_matrix_good_model(
            model=turbo_model,
            loader=camera_turbo_loader,
            class_labels=camera_labels,
            title="Camera — turbo_model nou",
            use_tta=True,
        )
    )
else:
    print("Nu pot face matrice pentru turbo_model: lipsește modelul sau loaderul.")


# 4. Camera — ResNet18 final
if "camera_resnet18_final" in globals() and camera_final_loader is not None:
    results_good_confusions.append(
        show_confusion_matrix_good_model(
            model=camera_resnet18_final,
            loader=camera_final_loader,
            class_labels=camera_labels,
            title="Camera — ResNet18 final nou",
            use_tta=True,
        )
    )
else:
    print("Nu pot face matrice pentru camera_resnet18_final: lipsește modelul sau loaderul.")


# 5. Camera — MobileNetV2 final
if "camera_mobilenetv2_final" in globals() and camera_final_loader is not None:
    results_good_confusions.append(
        show_confusion_matrix_good_model(
            model=camera_mobilenetv2_final,
            loader=camera_final_loader,
            class_labels=camera_labels,
            title="Camera — MobileNetV2 final nou",
            use_tta=True,
        )
    )
else:
    print("Nu pot face matrice pentru camera_mobilenetv2_final: lipsește modelul sau loaderul.")


results_good_df = pd.DataFrame(results_good_confusions)

print("\n" + "=" * 90)
print("REZUMAT MATRICE DE CONFUZIE — MODELE NOI BUNE")
print("=" * 90)

if len(results_good_df) > 0:
    display(results_good_df)

    summary_path = CM_DIR / "rezumat_matrici_modele_bune.csv"
    results_good_df.to_csv(summary_path, index=False)

    print("Rezumat salvat aici:")
    print(summary_path)
    display(FileLink(str(summary_path)))

    print("\nPNG-uri generate:")
    for p in sorted(CM_DIR.glob("*.png")):
        print(p.name)
        display(FileLink(str(p)))
else:
    print("Nu s-a generat nicio matrice. Probabil modelele noi nu mai sunt în memorie.")